In [1]:
from timescale import TimescaleDBManager
from dotenv import load_dotenv
from sqlalchemy import create_engine
import pandas as pd
import os
import matplotlib.pyplot as plt
import numpy as np
import joblib
import json
import re

from sklearn.model_selection import train_test_split, GridSearchCV
from sklearn.ensemble import IsolationForest, RandomForestClassifier
from sklearn.metrics import (
    confusion_matrix, ConfusionMatrixDisplay, accuracy_score, 
    precision_score, recall_score, f1_score, classification_report,
    roc_auc_score, roc_curve, RocCurveDisplay
)
from sklearn.inspection import permutation_importance
from sklearn.preprocessing import LabelEncoder, StandardScaler

from tensorflow.keras import layers, models
from tensorflow.keras.models import Model, Sequential, load_model
from tensorflow.keras.layers import Input, Dense, BatchNormalization, Dropout
from tensorflow.keras.optimizers import Adam
from tensorflow.keras.callbacks import Callback, EarlyStopping, ReduceLROnPlateau

from scikeras.wrappers import KerasRegressor
from xgboost import XGBClassifier
from lightgbm import LGBMClassifier


import optuna
from optuna.samplers import TPESampler
import tensorflow as tf
import seaborn as sns
from functools import partial
from tqdm import tqdm
import gc
import sys
import warnings


In [2]:
load_dotenv()
user_env        = os.getenv("USER_DB")
pass_env        = os.getenv("PASS_DB")
host_env        = os.getenv("HOST_DB", "localhost")
port_env        = os.getenv("DB_PORT_EXPOSED", "5432")
db_env          = os.getenv("NAME_DB", "tfm_db")
FOLDER_SALIDA   = os.getenv("FOLDER_SALIDA")
FOLDER_TMP      = os.getenv("FOLDER_TMP")

SEED            = 100#int(os.getenv("SEED"))

db = TimescaleDBManager(
    user=user_env,
    password=pass_env,
    host=host_env,
    port=port_env,
    dbname=db_env
)
db.conectar()
conexion_postgresql = db.connection

#esto es para usar una conexion postgrews para pandas que da un warning indicando que solo esta testeado para sqlalchemy
engine = create_engine(f'postgresql+psycopg2://{user_env}:{pass_env}@{host_env}:{port_env}/{db_env}')

[OK] Conexión establecida con TimescaleDB.


In [3]:
tabla_origen = 'vectores_logs'
tabla_split  = 'vectores_logs_split'
filename_lista_metric_name_vector = f'{FOLDER_SALIDA}/lista_metricas_filtradas_ord(metricas_logs).json'

In [4]:
SQL=f'select * from {tabla_origen}'
df = pd.read_sql(SQL, engine)


In [5]:
lista_label = df['label'].unique().tolist()

In [6]:
len(lista_label)

54

In [7]:
df_result=df[ ['vector_id','execution_name','label'] ].copy()


In [8]:
df_result

,vector_id,execution_name,label
0,115150,light-oauth2-data-1719592986.tar,correct
1,115151,light-oauth2-data-1719592986.tar,correct
2,115152,light-oauth2-data-1719592986.tar,correct
3,115153,light-oauth2-data-1719592986.tar,correct
4,115154,light-oauth2-data-1719592986.tar,correct
...,...,...,...
44121,159271,light-oauth2-data-1722744859.tar,get_user_404_no_user
44122,159272,light-oauth2-data-1722744859.tar,delete_user_404_no_user
44123,159273,light-oauth2-data-1722744859.tar,update_password_401_wrong_password
44124,159274,light-oauth2-data-1722744859.tar,update_password_404_user_not_found


In [9]:
with open(filename_lista_metric_name_vector, "r", encoding="utf-8") as f:
    lista_metric_name_vector = json.load(f)

In [10]:
len(lista_metric_name_vector)

331

In [11]:
def reporte_uso_memoria():

    # Lista para almacenar la info y poder ordenarla
    variables_info = []
    
    for nombre, obj in list(globals().items()):
        # Evitamos variables del sistema que empiezan por '_'
        if not nombre.startswith('_'):
            peso_mb = sys.getsizeof(obj) / (1024 * 1024)
            
            # Cálculo más preciso para DataFrames de pandas o matrices de numpy
            if hasattr(obj, 'memory_usage'):
                try:
                    peso_mb = obj.memory_usage(deep=True).sum() / (1024 * 1024)
                except:
                    pass
            elif hasattr(obj, 'nbytes'):
                peso_mb = obj.nbytes / (1024 * 1024)
                
            variables_info.append({
                'nombre': nombre,
                'tipo': type(obj).__name__,
                'peso': peso_mb
            })
    
    # Ordenar de mayor a menor peso (descendente)
    variables_info = sorted(variables_info, key=lambda x: x['peso'], reverse=True)
    
    # Imprimir la tabla formateada
    print(f"{'Variable':<30} | {'Tipo':<20} | {'Tamaño (MB)':<12}")
    print("-" * 68)
    for var in variables_info:
        print(f"{var['nombre']:<30} | {var['tipo']:<20} | {var['peso']:>10.2f} MB")

In [12]:


def X_matrix_sel_data(X_matrix,lista_feature_names,lista_idx_filas=None,lista_feature_sel=None):
    #X_matrix es la matriz completa
    #lista_feature_names son los nombres de las columnas
    #lista_idx_filas es la lista de indices de las filas que tomaremos
    #lista_feature_sel es la lista de nombres de las columnas que tomaremos, es un subconjunto de lista_feature_names
    if lista_idx_filas is None:
        lista_idx_filas   = range(0,X_matrix.shape[0])
    if lista_feature_sel is None:
        lista_feature_sel = lista_feature_names.copy()

    lista_feature_idx = []
    idx=0
    for feature in lista_feature_names:
        if feature in lista_feature_sel:
            lista_feature_idx.append(idx)
        idx += 1        

    # 1. Filtramos primero las filas
    X_temp = X_matrix[list(lista_idx_filas), :]
    
    # 2. Filtramos después las columnas sobre el resultado anterior
    X_matrix_sub = X_temp[:, lista_feature_idx]
    
    return X_matrix_sub

    


In [13]:
def evaluar_modelo(paquete_modelo,lista_feature_names,X_test,Y_test_etiquetas,encoder_etiquetas):
    #lista_feature_names  es el nombre de las columnas de X_test
    lista_feature_names_modelo = paquete_modelo['lista_features']
    model = paquete_modelo['model']
    lista_feature_idx = []
    idx=0
    for feature in lista_feature_names:
        if feature in lista_feature_names_modelo:
            lista_feature_idx.append(idx )
        idx += 1
        
    X_test_sub                                  = X_test[:, lista_feature_idx]
    Y_pred                                      = model.predict(X_test_sub)
    Y_pred_tipos                                = encoder_etiquetas.inverse_transform(Y_pred)
    Y_pred_probabilidades                       = model.predict_proba(X_test_sub)
    reporte_dict                                = classification_report(Y_test_etiquetas, Y_pred_tipos, output_dict=True, zero_division=0)
    classification_report_str                   = classification_report(Y_test_etiquetas, Y_pred_tipos)    
    accuracy                                    = reporte_dict['accuracy']
    macro_f1                                    = reporte_dict['macro avg']['f1-score']
    macro_precision                             = reporte_dict['macro avg']['precision']
    macro_recall                                = reporte_dict['macro avg']['recall']

    resultado                                   = {'accuracy': accuracy,  'precision':macro_precision   , 'recall':macro_recall  , 'f1':macro_f1 }

    paquete_modelo['resultado']                 = resultado
    paquete_modelo['probabilidades']            = Y_pred_probabilidades
    paquete_modelo['reporte_dict']              = reporte_dict
    paquete_modelo['classification_report_str'] = classification_report_str
    
    return resultado,reporte_dict,Y_pred_probabilidades

def predict_modelo(paquete_modelo,lista_feature_names,X_test):
    lista_feature_names_modelo = paquete_modelo['lista_features']
    model = paquete_modelo['model']
    lista_feature_idx = []
    idx=0
    for feature in lista_feature_names:
        if feature in lista_feature_names_modelo:
            lista_feature_idx.append(idx )
        idx += 1

    X_test_sub                                  = X_test[:, lista_feature_idx]        
    Y_pred                                      = model.predict(X_test_sub)   
    return Y_pred

    

def get_lista_feature_sin_lag(lista_features):
    lista_features_result=[]
    for feature in lista_features:
        feature_limpia = re.sub(r'\(lag_\d+\)$', '', feature)
        if feature_limpia not in  lista_features_result:
            lista_features_result.append(feature_limpia)
    return lista_features_result

    

In [14]:
#para optimizacion con OPTUNA
def print_callback(study, trial):
    # Si el trial falló por cualquier motivo, lo saltamos visualmente
    if trial.state != optuna.trial.TrialState.COMPLETE:
        pbar.update(1)
        return

    current_acc = trial.user_attrs.get('accuracy', trial.value)
    current_params = trial.params
    
    # Línea 1: Información de la iteración actual
    print(f"Trial {trial.number:2d}                | Actual Acc: {current_acc:.4f} | Parámetros: {current_params}")
    
    # Línea 2: Mejor resultado histórico hasta ahora
    print(f"   -> MEJOR HASTA AHORA | Mejor Acc:  {study.best_value:.4f} | Parámetros: {study.best_params}")
    print("-" * 100)
    
    # Actualización de la barra de progreso
    pbar.update(1)
    pbar.set_postfix({
        "Mejor_Acc": f"{study.best_value:.4f}",
        "Trial": trial.number
    })


In [15]:
def objective_universal(trial, config, clasificador_cls, X_train, X_test, Y_train_encoded, Y_test_encoded):
    params = {}
    
    for key, values in config.items():
        params[key] = trial.suggest_categorical(key, values)

    params['random_state'] = SEED
    if 'n_jobs' in clasificador_cls().get_params():
        params['n_jobs'] = -1

    model = clasificador_cls(**params)
    model.fit(X_train, Y_train_encoded)
    
    preds = model.predict(X_test)
    accuracy = accuracy_score(Y_test_encoded, preds)
    
    trial.set_user_attr('accuracy', accuracy)
    return accuracy

In [16]:
def obtener_importancias_agrupadas(importancias, lista_features):
    """
    Procesa las importancias de un modelo, elimina los sufijos de lag/sum,
    las agrupa por característica base y calcula cuántas características tienen importancia cero.
    """
    # 1. Crear el DataFrame inicial con las características y sus importancias
    df_importancias = pd.DataFrame({
        'caracteristica_con_lag': lista_features,
        'importancia': importancias
    }).sort_values(by='importancia', ascending=False).reset_index(drop=True)
    
    # 2. Limpiar los sufijos de lags y sumas para obtener la característica base
    df_importancias['caracteristica'] = (
        df_importancias['caracteristica_con_lag']
        .str.replace(r'\(lag_\d+\)', '', regex=True)
        .str.replace(r'\(sum\)', '', regex=False)
    )
    
    # 3. Agrupar por característica base sumando sus importancias
    df_importancias_agrupada = (
        df_importancias.groupby('caracteristica', as_index=False)['importancia']
        .sum()
        .sort_values(by='importancia', ascending=False)
        .reset_index(drop=True)
    )
    
    # 4. Contar cuántas características agrupadas tienen una importancia igual a 0
    caracteristicas_cero = df_importancias_agrupada[df_importancias_agrupada['importancia'] == 0]
    
    return df_importancias_agrupada, importancias, len(caracteristicas_cero)

# <span style="color:#2ca02c">1. Recuperamos separación ya hecha en conjuntos de train, val y test</span>

In [17]:
SQL=f'select * from {tabla_split}'
df_split = pd.read_sql(SQL, engine)

In [18]:
df_split

,vector_id,split_type
0,115150,test
1,115151,train
2,115152,train
3,115153,train
4,115154,train
...,...,...
44121,159271,train
44122,159272,train
44123,159273,train
44124,159274,train


In [19]:
df_result = pd.merge(df_result, df_split, on="vector_id", how="inner")

In [20]:
df_result

,vector_id,execution_name,label,split_type
0,115150,light-oauth2-data-1719592986.tar,correct,test
1,115151,light-oauth2-data-1719592986.tar,correct,train
2,115152,light-oauth2-data-1719592986.tar,correct,train
3,115153,light-oauth2-data-1719592986.tar,correct,train
4,115154,light-oauth2-data-1719592986.tar,correct,train
...,...,...,...,...
44121,159271,light-oauth2-data-1722744859.tar,get_user_404_no_user,train
44122,159272,light-oauth2-data-1722744859.tar,delete_user_404_no_user,train
44123,159273,light-oauth2-data-1722744859.tar,update_password_401_wrong_password,train
44124,159274,light-oauth2-data-1722744859.tar,update_password_404_user_not_found,train


In [21]:
dic_idx_train = {}
#dic_idx_resto = {}
dic_idx_val   = {}
dic_idx_test  = {}

In [22]:
for label in lista_label:
    idx_train = df_result[  (df_result['label']==label) & (df_result['split_type'] == 'train') ].index.values
    idx_val   = df_result[  (df_result['label']==label) & (df_result['split_type'] == 'val'  ) ].index.values
    idx_test  = df_result[  (df_result['label']==label) & (df_result['split_type'] == 'test') ].index.values
    dic_idx_train[label] = idx_train
    dic_idx_val[label]   = idx_val
    dic_idx_test[label]  = idx_test

<span style="color:#2ca02c"> Convetimos los vectores del df en una matriz, seran los datos que pasemos a los algoritmo   </span>


In [23]:
lista_matrix        = []
lista_feature_names = []

X_matrix1 = np.array(df['vector'].tolist())
X_matrix1.shape


(44126, 993)

In [24]:
N_puntos=3
lista_features = []
lista_features_orig = lista_metric_name_vector.copy()
for lag in range(0,N_puntos):
    for feature in lista_features_orig:
        lista_features.append(f'{feature}(lag_{lag})')
lista_feature_names1 = lista_features.copy()
lista_feature_names.extend(lista_feature_names1)
lista_matrix.append(X_matrix1)






In [25]:
lista_features_names_3puntos = lista_features.copy()

In [26]:
num_columnas_total = X_matrix1.shape[1]
tercio_n = num_columnas_total // N_puntos

# Definimos los límites de los 3 tercios
inicio_1, fin_1 = 0, tercio_n
inicio_2, fin_2 = tercio_n, tercio_n * 2
inicio_3, fin_3 = tercio_n * 2, num_columnas_total  # Coge el resto hasta el final por si acaso

# 2. Extraemos cada tercio como una submatriz
t1 = X_matrix1[:, inicio_1:fin_1]
t2 = X_matrix1[:, inicio_2:fin_2]
t3 = X_matrix1[:, inicio_3:fin_3]

# 3. Sumamos las columnas homólogas de los tres tercios (columna i del tercio 1 + columna i del tercio 2 + columna i del tercio 3)
# Como t1, t2 y t3 tienen exactamente la misma forma (n_filas, tercio_n), la suma de NumPy los alinea automáticamente.
X_matrix1_sum = t1 + t2 + t3
del t1,t2,t3
gc.collect()

0

In [27]:
lista_feature_names1_sum = []
for feature in lista_feature_names1:
    feature_sum = re.sub(r'\(lag_\d+\)', '(sum)', feature)
    if feature_sum not in lista_feature_names1_sum:
        lista_feature_names1_sum.append(feature_sum)


In [28]:
lista_matrix.append(X_matrix1_sum)
lista_feature_names.extend(lista_feature_names1_sum)

In [29]:
X_matrix = np.concatenate(lista_matrix, axis=1)

In [30]:
len(lista_feature_names)

1324

In [31]:
print(f'len(lista_feature_names1_sum)  = {len(lista_feature_names1_sum)}')
print(f'len(lista_feature_names1)      = {len(lista_feature_names1)}')
print(f'len(lista_feature_names)       = {len(lista_feature_names)}')

len(lista_feature_names1_sum)  = 331
len(lista_feature_names1)      = 993
len(lista_feature_names)       = 1324


In [32]:
lista_feature_names_1punto_sum=lista_feature_names1_sum.copy()

In [33]:
len(lista_feature_names_1punto_sum)

331

# <span style="color:#2ca02c">2. Preparacion de datos para entrenamiento/test</span>

In [34]:
#1. Establecemos la semilla para tener reproducibilidad
rng = np.random.default_rng(SEED)

# 2. Primero calculamos los mínimos por clase asegurando barajado aleatorio previo si se desea
min_train = float('inf')
min_test  = float('inf')
min_val   = float('inf')

for label in lista_label:
    if label == 'correct':
        continue
    
    min_train = min(min_train, len(dic_idx_train[label]))
    min_test  = min(min_test,  len(dic_idx_test[label]))
    min_val   = min(min_val,   len(dic_idx_val[label]))

# 3. Inicializamos los arrays de índices vacíos
idx_train = np.array([], dtype=int)
idx_test  = np.array([], dtype=int)
idx_val   = np.array([], dtype=int)

# 4. Bucle de selección aleatoria balanceada usando la SEED
for label in lista_label:
    if label == 'correct':
        continue
    
    # Barajamos los índices de forma aleatoria con la semilla antes de recortar
    arr_train = rng.permutation(dic_idx_train[label])
    arr_test  = rng.permutation(dic_idx_test[label])
    arr_val   = rng.permutation(dic_idx_val[label])
    
    # Concatenamos la cantidad mínima común de forma aleatoria, todas las clases tienen la misma cantidad de datos para que esten balanceadas
    idx_train = np.concatenate((idx_train, arr_train[:min_train]))
    idx_test  = np.concatenate((idx_test,  arr_test[:min_test]))
    idx_val   = np.concatenate((idx_val,   arr_val[:min_val]))

In [35]:
X_train = X_matrix_sel_data(X_matrix,lista_feature_names,lista_idx_filas=idx_train,lista_feature_sel=lista_feature_names1)
print(X_train.shape)
X_test  = X_matrix_sel_data(X_matrix,lista_feature_names,lista_idx_filas=idx_test ,lista_feature_sel=lista_feature_names1)
print(X_test.shape)
X_val   = X_matrix_sel_data(X_matrix,lista_feature_names,lista_idx_filas=idx_test ,lista_feature_sel=lista_feature_names1)
print(X_val.shape)

(10282, 993)
(3445, 993)
(3445, 993)


In [36]:
X_train_sum = X_matrix_sel_data(X_matrix,lista_feature_names,lista_idx_filas=idx_train,lista_feature_sel=lista_feature_names1_sum)
print(X_train_sum.shape)
X_test_sum  = X_matrix_sel_data(X_matrix,lista_feature_names,lista_idx_filas=idx_test ,lista_feature_sel=lista_feature_names1_sum)
print(X_test_sum.shape)
X_val_sum   = X_matrix_sel_data(X_matrix,lista_feature_names,lista_idx_filas=idx_test ,lista_feature_sel=lista_feature_names1_sum)
print(X_val_sum.shape)

(10282, 331)
(3445, 331)
(3445, 331)


In [37]:
X_matrix.shape

(44126, 1324)

In [38]:
df_result

,vector_id,execution_name,label,split_type
0,115150,light-oauth2-data-1719592986.tar,correct,test
1,115151,light-oauth2-data-1719592986.tar,correct,train
2,115152,light-oauth2-data-1719592986.tar,correct,train
3,115153,light-oauth2-data-1719592986.tar,correct,train
4,115154,light-oauth2-data-1719592986.tar,correct,train
...,...,...,...,...
44121,159271,light-oauth2-data-1722744859.tar,get_user_404_no_user,train
44122,159272,light-oauth2-data-1722744859.tar,delete_user_404_no_user,train
44123,159273,light-oauth2-data-1722744859.tar,update_password_401_wrong_password,train
44124,159274,light-oauth2-data-1722744859.tar,update_password_404_user_not_found,train


In [39]:
#esta parte hay que hacerla si no hemos crado antes el label encoder, si lo hemos creado , recuperamos el label encoder de fichero
# encoder_etiquetas = LabelEncoder()

ruta_encoder = f'{FOLDER_SALIDA}/models/encoder_etiquetas.joblib'
encoder_etiquetas = joblib.load(ruta_encoder)


for indice, etiqueta in enumerate(encoder_etiquetas.classes_):
    print(f"Índice {indice} --> {etiqueta}")

Índice 0 --> access_token_auth_header_error_401
Índice 1 --> access_token_authorization_form_401
Índice 2 --> access_token_client_id_not_found_404
Índice 3 --> access_token_client_secret_wrong_401
Índice 4 --> access_token_form_urlencoded_400
Índice 5 --> access_token_illegal_grant_type_400
Índice 6 --> access_token_missing_authorization_header_400
Índice 7 --> authorization_code_client_id_missing_400
Índice 8 --> authorization_code_invalid_client_id_404
Índice 9 --> authorization_code_invalid_password_401
Índice 10 --> authorization_code_missing_response_type_400
Índice 11 --> authorization_code_response_not_code_400
Índice 12 --> code_challenge_invalid_format_pkce_400
Índice 13 --> code_challenge_too_long_pkce_400
Índice 14 --> code_challenge_too_short_pkce_400
Índice 15 --> code_verifier_missing_pkce_400
Índice 16 --> code_verifier_too_long_pkce_400
Índice 17 --> code_verifier_too_short_pkce_400
Índice 18 --> delete_client_404_no_client
Índice 19 --> delete_service_404_no_service
Ín

In [40]:
Y_train_etiquetas = df.loc[idx_train , 'label']
Y_test_etiquetas  = df.loc[idx_test  , 'label']
Y_val_etiquetas   = df.loc[idx_val  , 'label']

Y_train_encoded = encoder_etiquetas.transform(Y_train_etiquetas)
Y_test_encoded  = encoder_etiquetas.transform(Y_test_etiquetas)
Y_val_encoded   = encoder_etiquetas.transform(Y_val_etiquetas)

In [41]:
Y_train_encoded

array([32, 32, 32, ...,  8,  8,  8], shape=(10282,))

In [42]:
reporte_uso_memoria()

Variable                       | Tipo                 | Tamaño (MB) 
--------------------------------------------------------------------
X_matrix                       | ndarray              |     445.73 MB
df                             | DataFrame            |     380.93 MB
X_matrix1                      | ndarray              |     334.30 MB
X_matrix1_sum                  | ndarray              |     111.43 MB
X_train                        | ndarray              |      77.90 MB
X_test                         | ndarray              |      26.10 MB
X_val                          | ndarray              |      26.10 MB
X_train_sum                    | ndarray              |      25.97 MB
df_result                      | DataFrame            |      10.16 MB
X_test_sum                     | ndarray              |       8.70 MB
X_val_sum                      | ndarray              |       8.70 MB
df_split                       | DataFrame            |       2.92 MB
Y_train_etiquetas     

# <span style="color:#2ca02c">3. N/A</span>

# <span style="color:#2ca02c">4. Clasificacion de anomalias</span>

## <span style="color:#2ca02c">4.1 Clasificador Random Forest con metricas_logs</span>

In [44]:
ALGORITMO_CLASIFICADOR = RandomForestClassifier
nombre_algoritmo       = 'RandomForest'

### <span style="color:#2ca02c">4.1.1 Clasificador Random Forest (3 puntos) con metricas_logs</span>

In [444]:
N_puntos = 3
DATOS    = 'metricas_logs 3 puntos'

In [507]:
len(lista_feature_names)

1324

In [554]:
#hacemos esto para preparar los datos en las mismas variables todos los algoritmos

lista_features = lista_features_names_3puntos

X_train = X_matrix_sel_data(X_matrix,lista_feature_names,lista_idx_filas=idx_train,lista_feature_sel=lista_features)
print(X_train.shape)
X_test  = X_matrix_sel_data(X_matrix,lista_feature_names,lista_idx_filas=idx_test ,lista_feature_sel=lista_features)
print(X_test.shape)
X_val   = X_matrix_sel_data(X_matrix,lista_feature_names,lista_idx_filas=idx_val ,lista_feature_sel=lista_features)
print(X_val.shape)

(10282, 993)
(3445, 993)
(3445, 993)


In [448]:
params_grid = {
    'n_estimators': [100, 200, 300, 500],
    'max_depth': [None, 10, 20, 30, 40],
    'min_samples_split': [2, 5, 10],
    'min_samples_leaf': [1, 2, 4],
    'max_features': ['sqrt', 'log2', 0.5],
    'class_weight': [None, 'balanced', 'balanced_subsample']
}

In [449]:
optuna.logging.set_verbosity(optuna.logging.WARNING)
N_trials = 30

# Barra de progreso global
pbar = tqdm(total=N_trials, desc=f"Optimizando {nombre_algoritmo}")

# Crear el estudio

study = optuna.create_study(
    direction='maximize',
    sampler=TPESampler(seed=SEED) 
)

# Enlazamos DIRECTAMENTE 'objective_universal' en el partial (evita usar alias antiguos)
objective_func = partial(
    objective_universal, 
    config=params_grid,
    clasificador_cls=ALGORITMO_CLASIFICADOR,
    X_train=X_train,
    X_test=X_test,
    Y_train_encoded=Y_train_encoded,
    Y_test_encoded=Y_test_encoded
)

# Ejecutar la optimización
print(f"=== INICIANDO OPTIMIZACIÓN DE {nombre_algoritmo} CON OPTUNA ===")
try:
    study.optimize(objective_func, n_trials=N_trials, callbacks=[print_callback], n_jobs=1)
finally:
    pbar.close()

# Resultados finales
print(f"\n=== MEJOR COMBINACIÓN ENCONTRADA ({nombre_algoritmo}) ===")
print(study.best_params)
print(f"Mejor Accuracy: {study.best_value:.4f}")

# Reconstruir el modelo final con los mejores parámetros óptimos
mejores_params = study.best_params.copy()
mejores_params['random_state'] = SEED
if 'n_jobs' in ALGORITMO_CLASIFICADOR().get_params():
    mejores_params['n_jobs'] = -1

Optimizando RandomForest:   0%|          | 0/30 [00:00<?, ?it/s]

=== INICIANDO OPTIMIZACIÓN DE RandomForest CON OPTUNA ===


Optimizando RandomForest:   3%|▎         | 1/30 [00:05<02:33,  5.29s/it, Mejor_Acc=0.5997, Trial=0]

Trial  0                | Actual Acc: 0.5997 | Parámetros: {'n_estimators': 500, 'max_depth': 30, 'min_samples_split': 5, 'min_samples_leaf': 4, 'max_features': 'sqrt', 'class_weight': None}
   -> MEJOR HASTA AHORA | Mejor Acc:  0.5997 | Parámetros: {'n_estimators': 500, 'max_depth': 30, 'min_samples_split': 5, 'min_samples_leaf': 4, 'max_features': 'sqrt', 'class_weight': None}
----------------------------------------------------------------------------------------------------


Optimizando RandomForest:   7%|▋         | 2/30 [00:06<01:21,  2.90s/it, Mejor_Acc=0.5997, Trial=1]

...

Trial 27                | Actual Acc: 0.5112 | Parámetros: {'n_estimators': 500, 'max_depth': None, 'min_samples_split': 5, 'min_samples_leaf': 1, 'max_features': 'log2', 'class_weight': 'balanced'}
   -> MEJOR HASTA AHORA | Mejor Acc:  0.6232 | Parámetros: {'n_estimators': 200, 'max_depth': None, 'min_samples_split': 5, 'min_samples_leaf': 1, 'max_features': 0.5, 'class_weight': 'balanced'}
----------------------------------------------------------------------------------------------------


Optimizando RandomForest:  97%|█████████▋| 29/30 [10:29<00:16, 16.39s/it, Mejor_Acc=0.6232, Trial=28]

Trial 28                | Actual Acc: 0.5678 | Parámetros: {'n_estimators': 100, 'max_depth': None, 'min_samples_split': 2, 'min_samples_leaf': 2, 'max_features': 'sqrt', 'class_weight': 'balanced'}
   -> MEJOR HASTA AHORA | Mejor Acc:  0.6232 | Parámetros: {'n_estimators': 200, 'max_depth': None, 'min_samples_split': 5, 'min_samples_leaf': 1, 'max_features': 0.5, 'class_weight': 'balanced'}
----------------------------------------------------------------------------------------------------


Optimizando RandomForest: 100%|██████████| 30/30 [11:35<00:00, 23.19s/it, Mejor_Acc=0.6232, Trial=29]

Trial 29                | Actual Acc: 0.5977 | Parámetros: {'n_estimators': 500, 'max_depth': 30, 'min_samples_split': 5, 'min_samples_leaf': 1, 'max_features': 0.5, 'class_weight': None}
   -> MEJOR HASTA AHORA | Mejor Acc:  0.6232 | Parámetros: {'n_estimators': 200, 'max_depth': None, 'min_samples_split': 5, 'min_samples_leaf': 1, 'max_features': 0.5, 'class_weight': 'balanced'}
----------------------------------------------------------------------------------------------------

=== MEJOR COMBINACIÓN ENCONTRADA (RandomForest) ===
{'n_estimators': 200, 'max_depth': None, 'min_samples_split': 5, 'min_samples_leaf': 1, 'max_features': 0.5, 'class_weight': 'balanced'}
Mejor Accuracy: 0.6232


In [485]:
clf_anomalias_opt = ALGORITMO_CLASIFICADOR(**mejores_params)
clf_anomalias_opt.fit(X_train, Y_train_encoded)

,"n_estimators n_estimators: int, default=100The number of trees in the forest... versionchanged:: 0.22 The default value of ``n_estimators`` changed from 10 to 100 in 0.22.",200
,"min_samples_split min_samples_split: int or float, default=2The minimum number of samples required to split an internal node:- If int, then consider `min_samples_split` as the minimum number.- If float, then `min_samples_split` is a fraction and `ceil(min_samples_split * n_samples)` are the minimum number of samples for each split... versionchanged:: 0.18 Added float values for fractions.",5
,"max_features max_features: {""sqrt"", ""log2"", None}, int or float, default=""sqrt""The number of features to consider when looking for the best split:- If int, then consider `max_features` features at each split.- If float, then `max_features` is a fraction and `max(1, int(max_features * n_features_in_))` features are considered at each split.- If ""sqrt"", then `max_features=sqrt(n_features)`.- If ""log2"", then `max_features=log2(n_features)`.- If None, then `max_features=n_features`... versionchanged:: 1.1 The default of `max_features` changed from `""auto""` to `""sqrt""`.Note: the search for a split does not stop until at least onevalid partition of the node samples is found, even if it requires toeffectively inspect more than ``max_features`` features.",0.5
,"n_jobs n_jobs: int, default=NoneThe number of jobs to run in parallel. :meth:`fit`, :meth:`predict`,:meth:`decision_path` and :meth:`apply` are all parallelized over thetrees. ``None`` means 1 unless in a :obj:`joblib.parallel_backend`context. ``-1`` means using all processors. See :term:`Glossary<n_jobs>` for more details.",-1
,"random_state random_state: int, RandomState instance or None, default=NoneControls both the randomness of the bootstrapping of the samples usedwhen building trees (if ``bootstrap=True``) and the sampling of thefeatures to consider when looking for the best split at each node(if ``max_features < n_features``).See :term:`Glossary <random_state>` for details.",100
,"class_weight class_weight: {""balanced"", ""balanced_subsample""}, dict or list of dicts, default=NoneWeights associated with classes in the form ``{class_label: weight}``.If not given, all classes are supposed to have weight one. Formulti-output problems, a list of dicts can be provided in the sameorder as the columns of y.Note that for multioutput (including multilabel) weights should bedefined for each class of every column in its own dict. For example,for four-class multilabel classification weights should be[{0: 1, 1: 1}, {0: 1, 1: 5}, {0: 1, 1: 1}, {0: 1, 1: 1}] instead of[{1:1}, {2:5}, {3:1}, {4:1}].The ""balanced"" mode uses the values of y to automatically adjustweights inversely proportional to class frequencies in the input dataas ``n_samples / (n_classes * np.bincount(y))``The ""balanced_subsample"" mode is the same as ""balanced"" except thatweights are computed based on the bootstrap sample for every treegrown.For multi-output, the weights of each column of y will be multiplied.Note that these weights will be multiplied with sample_weight (passedthrough the fit method) if sample_weight is specified.",'balanced'
,"criterion criterion: {""gini"", ""entropy"", ""log_loss""}, default=""gini""The function to measure the quality of a split. Supported criteria are""gini"" for the Gini impurity and ""log_loss"" and ""entropy"" both for theShannon information gain, see :ref:`tree_mathematical_formulation`.Note: This parameter is tree-specific.",'gini'
,"max_depth max_depth: int, default=NoneThe maximum depth of the tree. If None, then nodes are expanded untilall leaves are pure or until all leaves contain less thanmin_samples_split samples.",None
,"min_samples_leaf min_samples_leaf: int or float, default=1The minimum number of samples required to be at a leaf node.A split point at any depth will only be considered if it leaves atleast ``min_samples_leaf`` training samples in each of the left andright branche

In [486]:

model_name=f'{nombre_algoritmo} ({DATOS})(todas las features)'
paquete_modelo={}

paquete_modelo['model_name']         = model_name
paquete_modelo['model']              = clf_anomalias_opt
paquete_modelo['parametros']         = mejores_params
paquete_modelo['lista_features']     = lista_features
paquete_modelo['N_features']         = len(paquete_modelo['lista_features'] )
paquete_modelo['N_features_unicas']  = len(get_lista_feature_sin_lag(paquete_modelo['lista_features'] ))
paquete_modelo['datos']              = DATOS
paquete_modelo['algoritmo']          = nombre_algoritmo

resultado,reporte_dict,Y_pred_probabilidades = evaluar_modelo(paquete_modelo,lista_features,X_test,Y_test_etiquetas,encoder_etiquetas)
print(f"=== REPORTE DE CLASIFICACIÓN DE TIPOS DE ANOMALÍA ===\n{model_name}")
print(paquete_modelo['classification_report_str'])

joblib.dump(paquete_modelo, f'{FOLDER_SALIDA}/models/{model_name}.joblib')

=== REPORTE DE CLASIFICACIÓN DE TIPOS DE ANOMALÍA ===
RandomForest (metricas_logs 3 puntos)(todas las features)
                                               precision    recall  f1-score   support

           access_token_auth_header_error_401       1.00      0.97      0.98        65
          access_token_authorization_form_401       0.43      0.58      0.49        65
         access_token_client_id_not_found_404       0.98      0.94      0.96        65
         access_token_client_secret_wrong_401       0.98      0.97      0.98        65
             access_token_form_urlencoded_400       0.61      0.57      0.59        65
          access_token_illegal_grant_type_400       0.97      0.98      0.98        65
access_token_missing_authorization_header_400       0.49      0.43      0.46        65
     authorization_code_client_id_missing_400       0.57      0.71      0.63        65
     authorization_code_invalid_client_id_404       0.63      0.65      0.64        65
      authorizati

['result/models/RandomForest (metricas_logs 3 puntos)(todas las features).joblib']

In [495]:
df_importancias_agrupada, importancias, num_cero = obtener_importancias_agrupadas(paquete_modelo['model'].feature_importances_, paquete_modelo['lista_features'])
print(f"Características con importancia cero: {num_cero}")

Características con importancia cero: 35


(10282, 993)

In [555]:
n_min = 62
n_max = len(importancias) - num_cero
lista_result = []

# --- VARIABLES PARA EL EARLY STOPPING ---
paciencia = 15
iteraciones_sin_mejora = 0
mejor_score_global = -1.0
mejor_paquete_modelo = None

for N_feat in range(n_min, n_max + 1):
    model_name = f'{nombre_algoritmo} ({DATOS})   N_feat={N_feat}'
    lista_features_sel = []
    for feature in df_importancias_agrupada['caracteristica'].to_list()[:N_feat]:
        if N_puntos>1:
            for lag in range(0, N_puntos):
                lista_features_sel.append(f'{feature}(lag_{lag})')
        else:
            lista_features_sel.append(feature)

    X_train_sub = X_matrix_sel_data(X_matrix, lista_feature_names, lista_idx_filas=idx_train, lista_feature_sel=lista_features_sel)
    X_test_sub  = X_matrix_sel_data(X_matrix, lista_feature_names, lista_idx_filas=idx_test,  lista_feature_sel=lista_features_sel)
   
    clf_anomalias_opt = ALGORITMO_CLASIFICADOR(**mejores_params)
    clf_anomalias_opt.fit(X_train_sub, Y_train_encoded)

    paquete_modelo = {}
    paquete_modelo['model_name']          = model_name
    paquete_modelo['model']               = clf_anomalias_opt
    paquete_modelo['parametros']          = mejores_params
    paquete_modelo['lista_features']      = lista_features_sel
    paquete_modelo['N_features']          = len(paquete_modelo['lista_features'])
    paquete_modelo['N_features_unicas']   = len(get_lista_feature_sin_lag(paquete_modelo['lista_features']))
    paquete_modelo['datos']               = 'metricas_log_lags'
    paquete_modelo['algoritmo']           = nombre_algoritmo
    
    resultado, reporte_dict, Y_pred_probabilidades = evaluar_modelo(paquete_modelo, lista_features, X_test, Y_test_etiquetas, encoder_etiquetas)
    # accuracy        = resultado['accuracy']
    # macro_f1        = resultado['f1']
    # macro_precision = resultado['precision']
    # macro_recall    = resultado['recall']
    
    # paquete_modelo['metrics'] = {
    #     'accuracy': accuracy,
    #     'macro_f1': macro_f1,
    #     'macro_precision': macro_precision,
    #     'macro_recall': macro_recall
    # }
    
    lista_result.append(paquete_modelo)

    # --- LÓGICA DE EARLY STOPPING ---
    score_actual = macro_f1 
    
    if score_actual > mejor_score_global:
        mejor_score_global = score_actual
        mejor_paquete_modelo = paquete_modelo.copy()
        iteraciones_sin_mejora = 0  # Reiniciamos contador si mejora
    else:
        iteraciones_sin_mejora += 1

    # --- IMPRESIÓN EN UNA SOLA LÍNEA ---
    print(f"Model = {model_name:40s} Accuracy = {accuracy:.4f} | Macro F1 = {macro_f1:.4f} | Macro Precision = {macro_precision:.4f} | Macro Recall = {macro_recall:.4f} | Mejor Macro F1 = {mejor_score_global:.4f} | Iteraciones_sin_mejora = {iteraciones_sin_mejora}")

    # Comprobación de parada
    if iteraciones_sin_mejora >= paciencia:
        print(f"\n[!] Early Stopping activado: No ha habido mejoras en {paciencia} iteraciones consecutivas.")
        print(f"Deteniendo búsqueda. El mejor modelo se logró con {mejor_paquete_modelo['N_features_unicas']} características base.")
        break

ValueError: Found array with 0 feature(s) (shape=(3445, 0)) while a minimum of 1 is required by RandomForestClassifier.

In [546]:
# 1. Partimos del modelo en el que nos hemos parado (llevaba ya le numero de iteraciones paciencia sin mejorar)
paquete_modelo_back = lista_result[-1].copy()


In [547]:
paquete_modelo = mejor_paquete_modelo.copy()

In [550]:
lista_features_sel

[]

In [553]:
len(paquete_modelo['lista_features'])

43

In [548]:

paquete_modelo = mejor_paquete_modelo.copy()

mejor_macro_f1_global    = paquete_modelo['resultado']['f1']
mejor_features_global    = paquete_modelo['lista_features'].copy()
pasos_sin_mejora         = 0
limite_puntos_sin_mejora = paciencia + 5

print(f"Iniciando eliminación recursiva desde {paquete_modelo['N_features_unicas']} características.\n")

while paquete_modelo['N_features_unicas'] > 1 and pasos_sin_mejora < limite_puntos_sin_mejora:
    N_feat = paquete_modelo['N_features_unicas']
    
    # Extraer importancias y características actuales
    importancias_raw = paquete_modelo['model'].feature_importances_
    features_raw = paquete_modelo['lista_features']
    
    df_importancias_agrupada, _, num_cero = obtener_importancias_agrupadas(importancias_raw, features_raw)

    # Seleccionar características para el siguiente paso
    lista_features_sel = []
    for feature in df_importancias_agrupada['caracteristica'].to_list()[:(N_feat-1)]:
        if N_puntos>1:
            for lag in range(0, N_puntos):
                lista_features_sel.append(f'{feature}(lag_{lag})')
        else:
            lista_features_sel.append(f'{feature}')
    
    X_train_sub = X_matrix_sel_data(X_matrix, lista_feature_names, lista_idx_filas=idx_train, lista_feature_sel=lista_features_sel)
    X_test_sub  = X_matrix_sel_data(X_matrix, lista_feature_names, lista_idx_filas=idx_test,  lista_feature_sel=lista_features_sel)
   
    clf_anomalias_opt = ALGORITMO_CLASIFICADOR(**mejores_params)
    clf_anomalias_opt.fit(X_train_sub, Y_train_encoded)

    model_name = f'{nombre_algoritmo} (metricas_log) N_feat={N_feat-1}'
    paquete_modelo = {}
    paquete_modelo['model_name']          = model_name
    paquete_modelo['model']               = clf_anomalias_opt
    paquete_modelo['parametros']          = mejores_params
    paquete_modelo['lista_features']      = lista_features_sel
    paquete_modelo['N_features']          = len(paquete_modelo['lista_features'])
    paquete_modelo['N_features_unicas']   = len(get_lista_feature_sin_lag(paquete_modelo['lista_features']))
    paquete_modelo['datos']               = 'metricas_log_lags'
    paquete_modelo['algoritmo']           = nombre_algoritmo
    
    resultado, reporte_dict, Y_pred_probabilidades = evaluar_modelo(paquete_modelo, lista_features, X_test, Y_test_etiquetas, encoder_etiquetas)
   
    accuracy        = resultado['accuracy']
    macro_f1        = resultado['f1']
    macro_precision = resultado['precision']
    macro_recall    = resultado['recall']
    

    
    if macro_f1 >= mejor_macro_f1_global:
        mejor_macro_f1_global = macro_f1
        mejor_features_global = paquete_modelo['lista_features'].copy()
        pasos_sin_mejora = 0  
        mejor_paquete_modelo = paquete_modelo.copy()
    else:
        pasos_sin_mejora += 1  
        
    print(f"Model = {model_name:40s} Accuracy = {accuracy:.4f} | Macro F1 = {macro_f1:.4f} | Macro Precision = {macro_precision:.4f} | Macro Recall = {macro_recall:.4f} | Sin mejora: {pasos_sin_mejora}")
    del X_train_sub, X_test_sub, importancias_raw, df_importancias_agrupada
    gc.collect()

print(f"\n¡Proceso finalizado!")
print(f"Mejor Macro F1 alcanzado: {mejor_macro_f1_global:.4f}")
print(f"Número óptimo de características base: {len(mejor_features_global)}")

Iniciando eliminación recursiva desde 43 características.



ValueError: Found array with 0 feature(s) (shape=(10282, 0)) while a minimum of 1 is required by RandomForestClassifier.

In [501]:
mejor_paquete_modelo['model_name']

'RandomForest (metricas_logs 3 puntos)   N_feat=63'

In [502]:
joblib.dump(mejor_paquete_modelo, f"{FOLDER_SALIDA}/models/{mejor_paquete_modelo['model_name']}.joblib")

['result/models/RandomForest (metricas_logs 3 puntos)   N_feat=63.joblib']

### <span style="color:#2ca02c">4.1.2 Clasificador Random Forest (1 punto de 3 intervalos) con metricas_logs</span>

In [45]:
N_puntos = 1
DATOS    = 'metricas_logs 1 punto suma'

In [46]:
lista_features = lista_feature_names_1punto_sum

X_train = X_matrix_sel_data(X_matrix,lista_feature_names,lista_idx_filas=idx_train,lista_feature_sel=lista_features)
print(X_train.shape)
X_test  = X_matrix_sel_data(X_matrix,lista_feature_names,lista_idx_filas=idx_test ,lista_feature_sel=lista_features)
print(X_test.shape)
X_val   = X_matrix_sel_data(X_matrix,lista_feature_names,lista_idx_filas=idx_val ,lista_feature_sel=lista_features)
print(X_val.shape)

(10282, 331)
(3445, 331)
(3445, 331)


In [47]:
len(lista_features)

331

In [48]:
params_grid = {
    'n_estimators': [100, 200, 300, 500],
    'max_depth': [None, 10, 20, 30, 40],
    'min_samples_split': [2, 5, 10],
    'min_samples_leaf': [1, 2, 4],
    'max_features': ['sqrt', 'log2', 0.5],
    'class_weight': [None, 'balanced', 'balanced_subsample']
}

In [49]:
optuna.logging.set_verbosity(optuna.logging.WARNING)
N_trials = 30

# Barra de progreso global
pbar = tqdm(total=N_trials, desc=f"Optimizando {nombre_algoritmo}")

# Crear el estudio

study = optuna.create_study(
    direction='maximize',
    sampler=TPESampler(seed=SEED) 
)

# Enlazamos DIRECTAMENTE 'objective_universal' en el partial (evita usar alias antiguos)
objective_func = partial(
    objective_universal, 
    config=params_grid,
    clasificador_cls=ALGORITMO_CLASIFICADOR,
    X_train=X_train,
    X_test=X_test,
    Y_train_encoded=Y_train_encoded,
    Y_test_encoded=Y_test_encoded
)

# Ejecutar la optimización
print(f"=== INICIANDO OPTIMIZACIÓN DE {nombre_algoritmo} CON OPTUNA ===")
try:
    study.optimize(objective_func, n_trials=N_trials, callbacks=[print_callback], n_jobs=1)
finally:
    pbar.close()

# Resultados finales
print(f"\n=== MEJOR COMBINACIÓN ENCONTRADA ({nombre_algoritmo}) ===")
print(study.best_params)
print(f"Mejor Accuracy: {study.best_value:.4f}")

# Reconstruir el modelo final con los mejores parámetros óptimos
mejores_params = study.best_params.copy()
mejores_params['random_state'] = SEED
if 'n_jobs' in ALGORITMO_CLASIFICADOR().get_params():
    mejores_params['n_jobs'] = -1

Optimizando RandomForest:   0%|          | 0/30 [00:00<?, ?it/s]

=== INICIANDO OPTIMIZACIÓN DE RandomForest CON OPTUNA ===


Optimizando RandomForest:   3%|▎         | 1/30 [00:04<01:58,  4.07s/it, Mejor_Acc=0.6000, Trial=0]

Trial  0                | Actual Acc: 0.6000 | Parámetros: {'n_estimators': 500, 'max_depth': 30, 'min_samples_split': 5, 'min_samples_leaf': 4, 'max_features': 'sqrt', 'class_weight': None}
   -> MEJOR HASTA AHORA | Mejor Acc:  0.6000 | Parámetros: {'n_estimators': 500, 'max_depth': 30, 'min_samples_split': 5, 'min_samples_leaf': 4, 'max_features': 'sqrt', 'class_weight': None}
----------------------------------------------------------------------------------------------------


Optimizando RandomForest:   7%|▋         | 2/30 [00:05<01:04,  2.31s/it, Mejor_Acc=0.6000, Trial=1]

...

Trial 27                | Actual Acc: 0.5988 | Parámetros: {'n_estimators': 200, 'max_depth': 40, 'min_samples_split': 10, 'min_samples_leaf': 1, 'max_features': 'sqrt', 'class_weight': 'balanced'}
   -> MEJOR HASTA AHORA | Mejor Acc:  0.6136 | Parámetros: {'n_estimators': 500, 'max_depth': 40, 'min_samples_split': 10, 'min_samples_leaf': 1, 'max_features': 'sqrt', 'class_weight': 'balanced'}
----------------------------------------------------------------------------------------------------


Optimizando RandomForest:  97%|█████████▋| 29/30 [02:04<00:04,  4.17s/it, Mejor_Acc=0.6136, Trial=28]

Trial 28                | Actual Acc: 0.5919 | Parámetros: {'n_estimators': 100, 'max_depth': 40, 'min_samples_split': 2, 'min_samples_leaf': 2, 'max_features': 0.5, 'class_weight': 'balanced'}
   -> MEJOR HASTA AHORA | Mejor Acc:  0.6136 | Parámetros: {'n_estimators': 500, 'max_depth': 40, 'min_samples_split': 10, 'min_samples_leaf': 1, 'max_features': 'sqrt', 'class_weight': 'balanced'}
----------------------------------------------------------------------------------------------------


Optimizando RandomForest: 100%|██████████| 30/30 [02:06<00:00,  4.23s/it, Mejor_Acc=0.6136, Trial=29]

Trial 29                | Actual Acc: 0.5736 | Parámetros: {'n_estimators': 500, 'max_depth': 30, 'min_samples_split': 5, 'min_samples_leaf': 1, 'max_features': 'log2', 'class_weight': 'balanced'}
   -> MEJOR HASTA AHORA | Mejor Acc:  0.6136 | Parámetros: {'n_estimators': 500, 'max_depth': 40, 'min_samples_split': 10, 'min_samples_leaf': 1, 'max_features': 'sqrt', 'class_weight': 'balanced'}
----------------------------------------------------------------------------------------------------

=== MEJOR COMBINACIÓN ENCONTRADA (RandomForest) ===
{'n_estimators': 500, 'max_depth': 40, 'min_samples_split': 10, 'min_samples_leaf': 1, 'max_features': 'sqrt', 'class_weight': 'balanced'}
Mejor Accuracy: 0.6136


In [100]:
clf_anomalias_opt = ALGORITMO_CLASIFICADOR(**mejores_params)
clf_anomalias_opt.fit(X_train, Y_train_encoded)

,"n_estimators n_estimators: int, default=100The number of trees in the forest... versionchanged:: 0.22 The default value of ``n_estimators`` changed from 10 to 100 in 0.22.",500
,"max_depth max_depth: int, default=NoneThe maximum depth of the tree. If None, then nodes are expanded untilall leaves are pure or until all leaves contain less thanmin_samples_split samples.",40
,"min_samples_split min_samples_split: int or float, default=2The minimum number of samples required to split an internal node:- If int, then consider `min_samples_split` as the minimum number.- If float, then `min_samples_split` is a fraction and `ceil(min_samples_split * n_samples)` are the minimum number of samples for each split... versionchanged:: 0.18 Added float values for fractions.",10
,"n_jobs n_jobs: int, default=NoneThe number of jobs to run in parallel. :meth:`fit`, :meth:`predict`,:meth:`decision_path` and :meth:`apply` are all parallelized over thetrees. ``None`` means 1 unless in a :obj:`joblib.parallel_backend`context. ``-1`` means using all processors. See :term:`Glossary<n_jobs>` for more details.",-1
,"random_state random_state: int, RandomState instance or None, default=NoneControls both the randomness of the bootstrapping of the samples usedwhen building trees (if ``bootstrap=True``) and the sampling of thefeatures to consider when looking for the best split at each node(if ``max_features < n_features``).See :term:`Glossary <random_state>` for details.",100
,"class_weight class_weight: {""balanced"", ""balanced_subsample""}, dict or list of dicts, default=NoneWeights associated with classes in the form ``{class_label: weight}``.If not given, all classes are supposed to have weight one. Formulti-output problems, a list of dicts can be provided in the sameorder as the columns of y.Note that for multioutput (including multilabel) weights should bedefined for each class of every column in its own dict. For example,for four-class multilabel classification weights should be[{0: 1, 1: 1}, {0: 1, 1: 5}, {0: 1, 1: 1}, {0: 1, 1: 1}] instead of[{1:1}, {2:5}, {3:1}, {4:1}].The ""balanced"" mode uses the values of y to automatically adjustweights inversely proportional to class frequencies in the input dataas ``n_samples / (n_classes * np.bincount(y))``The ""balanced_subsample"" mode is the same as ""balanced"" except thatweights are computed based on the bootstrap sample for every treegrown.For multi-output, the weights of each column of y will be multiplied.Note that these weights will be multiplied with sample_weight (passedthrough the fit method) if sample_weight is specified.",'balanced'
,"criterion criterion: {""gini"", ""entropy"", ""log_loss""}, default=""gini""The function to measure the quality of a split. Supported criteria are""gini"" for the Gini impurity and ""log_loss"" and ""entropy"" both for theShannon information gain, see :ref:`tree_mathematical_formulation`.Note: This parameter is tree-specific.",'gini'
,"min_samples_leaf min_samples_leaf: int or float, default=1The minimum number of samples required to be at a leaf node.A split point at any depth will only be considered if it leaves atleast ``min_samples_leaf`` training samples in each of the left andright branches. This may have the effect of smoothing the model,especially in regression.- If int, then consider `min_samples_leaf` as the minimum number.- If float, then `min_samples_leaf` is a fraction and `ceil(min_samples_leaf * n_samples)` are the minimum number of samples for each node... versionchanged:: 0.18 Added float values for fractions.",1
,"min_weight_fraction_leaf min_weight_fraction_leaf: float, default=0.0The minimum weighted fraction of the sum total of weights (of allthe input samples) required to be at a leaf node. Samples haveequal weight when sample_weight is not provided.",0.0
,"max_features max_features: {""sqrt"", ""log2"", None}, int or float, default=""sqrt""The number of features to consider when looking for the best split:- If int, then consider `max_fe

In [101]:
model_name=f'{nombre_algoritmo} ({DATOS})(todas las features)'
paquete_modelo={}

paquete_modelo['model_name']         = model_name
paquete_modelo['model']              = clf_anomalias_opt
paquete_modelo['parametros']         = mejores_params
paquete_modelo['lista_features']     = lista_features
paquete_modelo['N_features']         = len(paquete_modelo['lista_features'] )
paquete_modelo['N_features_unicas']  = len(get_lista_feature_sin_lag(paquete_modelo['lista_features'] ))
paquete_modelo['datos']              = DATOS
paquete_modelo['algoritmo']          = nombre_algoritmo

resultado,reporte_dict,Y_pred_probabilidades = evaluar_modelo(paquete_modelo,lista_features,X_test,Y_test_etiquetas,encoder_etiquetas)
print(f"=== REPORTE DE CLASIFICACIÓN DE TIPOS DE ANOMALÍA ===\n{model_name}")
print(paquete_modelo['classification_report_str'])

joblib.dump(paquete_modelo, f'{FOLDER_SALIDA}/models/{model_name}.joblib')

=== REPORTE DE CLASIFICACIÓN DE TIPOS DE ANOMALÍA ===
RandomForest (metricas_logs 1 punto suma)(todas las features)
                                               precision    recall  f1-score   support

           access_token_auth_header_error_401       0.87      0.92      0.90        65
          access_token_authorization_form_401       0.44      0.83      0.57        65
         access_token_client_id_not_found_404       0.97      1.00      0.98        65
         access_token_client_secret_wrong_401       0.93      0.95      0.94        65
             access_token_form_urlencoded_400       0.68      0.66      0.67        65
          access_token_illegal_grant_type_400       1.00      1.00      1.00        65
access_token_missing_authorization_header_400       0.43      0.28      0.34        65
     authorization_code_client_id_missing_400       0.70      0.71      0.70        65
     authorization_code_invalid_client_id_404       0.60      0.66      0.63        65
      authori

['result/models/RandomForest (metricas_logs 1 punto suma)(todas las features).joblib']

In [102]:
df_importancias_agrupada, importancias, num_cero = obtener_importancias_agrupadas(paquete_modelo['model'].feature_importances_, paquete_modelo['lista_features'])
print(f"Características con importancia cero: {num_cero}")

Características con importancia cero: 30


In [103]:
df_importancias_agrupada

,caracteristica,importancia
0,light-oauth2-oauth2-user-1(0)(sum),0.033865
1,light-oauth2-oauth2-service-1(0)(sum),0.028705
2,light-oauth2-oauth2-code-1(0)(sum),0.022908
3,light-oauth2-oauth2-client-1(20)(sum),0.020397
4,light-oauth2-oauth2-client-1(14)(sum),0.020376
...,...,...
326,light-oauth2-oauth2-client-1(74)(sum),0.000000
327,light-oauth2-oauth2-client-1(69)(sum),0.000000
328,light-oauth2-oauth2-client-1(68)(sum),0.000000
329,light-oauth2-oauth2-token-1(99)(sum),0.000000


In [104]:
n_min = 78
n_max = len(importancias) - num_cero
lista_result = []

# --- VARIABLES PARA EL EARLY STOPPING ---
paciencia = 10
iteraciones_sin_mejora = 0
mejor_score_global = -1.0
mejor_paquete_modelo = None

for N_feat in range(n_min, n_max + 1):
    model_name = f'{nombre_algoritmo} ({DATOS})   N_feat={N_feat}'
    lista_features_sel = []
    for feature in df_importancias_agrupada['caracteristica'].to_list()[:N_feat]:
        if N_puntos>1:
            for lag in range(0, N_puntos):
                lista_features_sel.append(f'{feature}(lag_{lag})')
        else:
            lista_features_sel.append(feature)

    X_train_sub = X_matrix_sel_data(X_matrix, lista_feature_names, lista_idx_filas=idx_train, lista_feature_sel=lista_features_sel)
    X_test_sub  = X_matrix_sel_data(X_matrix, lista_feature_names, lista_idx_filas=idx_test,  lista_feature_sel=lista_features_sel)
   
    clf_anomalias_opt = ALGORITMO_CLASIFICADOR(**mejores_params)
    clf_anomalias_opt.fit(X_train_sub, Y_train_encoded)

    paquete_modelo = {}
    paquete_modelo['model_name']          = model_name
    paquete_modelo['model']               = clf_anomalias_opt
    paquete_modelo['parametros']          = mejores_params
    paquete_modelo['lista_features']      = lista_features_sel
    paquete_modelo['N_features']          = len(paquete_modelo['lista_features'])
    paquete_modelo['N_features_unicas']   = len(get_lista_feature_sin_lag(paquete_modelo['lista_features']))
    paquete_modelo['datos']               = 'metricas_log_lags'
    paquete_modelo['algoritmo']           = nombre_algoritmo
    
    resultado, reporte_dict, Y_pred_probabilidades = evaluar_modelo(paquete_modelo, lista_features, X_test, Y_test_etiquetas, encoder_etiquetas)
    # accuracy        = resultado['accuracy']
    # macro_f1        = resultado['f1']
    # macro_precision = resultado['precision']
    # macro_recall    = resultado['recall']
    
    # paquete_modelo['resultado'] = {
    #     'accuracy': accuracy,
    #     'macro_f1': macro_f1,
    #     'macro_precision': macro_precision,
    #     'macro_recall': macro_recall
    # }
    
    #lista_result.append(paquete_modelo)

    # --- LÓGICA DE EARLY STOPPING ---
    score_actual = macro_f1 
    
    if score_actual > mejor_score_global:
        mejor_score_global = score_actual
        mejor_paquete_modelo = paquete_modelo.copy()
        iteraciones_sin_mejora = 0  # Reiniciamos contador si mejora
    else:
        iteraciones_sin_mejora += 1

    # --- IMPRESIÓN EN UNA SOLA LÍNEA ---
    print(f"Model = {model_name:40s} Accuracy = {accuracy:.4f} | Macro F1 = {macro_f1:.4f} | Macro Precision = {macro_precision:.4f} | Macro Recall = {macro_recall:.4f} | Mejor Macro F1 = {mejor_score_global:.4f} | Iteraciones_sin_mejora = {iteraciones_sin_mejora}")
    del X_train_sub, X_test_sub
    gc.collect()
    
    # Comprobación de parada
    if iteraciones_sin_mejora >= paciencia:
        print(f"\n[!] Early Stopping activado: No ha habido mejoras en {paciencia} iteraciones consecutivas.")
        print(f"Deteniendo búsqueda. El mejor modelo se logró con {mejor_paquete_modelo['N_features_unicas']} características base.")
        break

Model = RandomForest (metricas_logs 1 punto suma)   N_feat=78 Accuracy = 0.6276 | Macro F1 = 0.6197 | Macro Precision = 0.6283 | Macro Recall = 0.6276 | Mejor Macro F1 = 0.6197 | Iteraciones_sin_mejora = 0
Model = RandomForest (metricas_logs 1 punto suma)   N_feat=79 Accuracy = 0.6186 | Macro F1 = 0.6105 | Macro Precision = 0.6183 | Macro Recall = 0.6186 | Mejor Macro F1 = 0.6197 | Iteraciones_sin_mejora = 1
Model = RandomForest (metricas_logs 1 punto suma)   N_feat=80 Accuracy = 0.6197 | Macro F1 = 0.6114 | Macro Precision = 0.6195 | Macro Recall = 0.6197 | Mejor Macro F1 = 0.6197 | Iteraciones_sin_mejora = 2
Model = RandomForest (metricas_logs 1 punto suma)   N_feat=81 Accuracy = 0.6183 | Macro F1 = 0.6111 | Macro Precision = 0.6198 | Macro Recall = 0.6183 | Mejor Macro F1 = 0.6197 | Iteraciones_sin_mejora = 3
Model = RandomForest (metricas_logs 1 punto suma)   N_feat=82 Accuracy = 0.6221 | Macro F1 = 0.6139 | Macro Precision = 0.6240 | Macro Recall = 0.6221 | Mejor Macro F1 = 0.6197

In [105]:
xxx=mejor_paquete_modelo.copy()

In [106]:
df_importancias_agrupada, importancias, num_cero = obtener_importancias_agrupadas(paquete_modelo['model'].feature_importances_, paquete_modelo['lista_features'])
print(f"Características con importancia cero: {num_cero}")

Características con importancia cero: 0


In [107]:
df_importancias_agrupada

,caracteristica,importancia
0,light-oauth2-oauth2-code-1(27)(sum),0.044427
1,light-oauth2-oauth2-service-1(10)(sum),0.037410
2,light-oauth2-oauth2-token-1(41)(sum),0.028990
3,light-oauth2-oauth2-user-1(6)(sum),0.024863
4,light-oauth2-oauth2-client-1(14)(sum),0.023616
...,...,...
83,light-oauth2-oauth2-client-1(11)(sum),0.006002
84,light-oauth2-oauth2-code-1(22)(sum),0.005994
85,light-oauth2-oauth2-user-1(7)(sum),0.005748
86,light-oauth2-oauth2-client-1(15)(sum),0.005483


In [116]:

paquete_modelo = mejor_paquete_modelo.copy()


# features_actuales = df_importancias_agrupada['caracteristica'].to_list()[:N_inicio]

mejor_macro_f1_global    = paquete_modelo['resultado']['macro_f1']
mejor_features_global    = paquete_modelo['lista_features'].copy()
pasos_sin_mejora         = 0
limite_puntos_sin_mejora = paciencia + 5

print(f"Iniciando eliminación recursiva desde {paquete_modelo['N_features_unicas']} características.\n")

while paquete_modelo['N_features_unicas'] > 1 and pasos_sin_mejora < limite_puntos_sin_mejora:

    df_importancias_agrupada, importancias, num_cero = obtener_importancias_agrupadas(paquete_modelo['model'].feature_importances_, paquete_modelo['lista_features'])
 
    lista_features_sel=[]
    for feature in df_importancias_agrupada['caracteristica'].to_list()[:(N_feat-1)]:
        if N_puntos>1:
            for lag in range(0, N_puntos):
                lista_features_sel.append(f'{feature}(lag_{lag})')
        else:
            lista_features_sel.append(f'{feature}')

    
    X_train_sub = X_matrix_sel_data(X_matrix, lista_feature_names, lista_idx_filas=idx_train, lista_feature_sel=lista_features_sel)
    X_test_sub  = X_matrix_sel_data(X_matrix, lista_feature_names, lista_idx_filas=idx_test,  lista_feature_sel=lista_features_sel)
   
    clf_anomalias_opt = ALGORITMO_CLASIFICADOR(**mejores_params)
    clf_anomalias_opt.fit(X_train_sub, Y_train_encoded)
    
    N_feat = len(lista_features_sel)
    model_name = f'{nombre_algoritmo} ({DATOS})   N_feat={N_feat}'

    paquete_modelo = {}
    paquete_modelo['model_name']          = model_name
    paquete_modelo['model']               = clf_anomalias_opt
    paquete_modelo['parametros']          = mejores_params
    paquete_modelo['lista_features']      = lista_features_sel
    paquete_modelo['N_features']          = len(paquete_modelo['lista_features'])
    paquete_modelo['N_features_unicas']   = len(get_lista_feature_sin_lag(paquete_modelo['lista_features']))
    paquete_modelo['datos']               = 'metricas_log_lags'
    paquete_modelo['algoritmo']           = nombre_algoritmo
    

    resultado, reporte_dict, Y_pred_probabilidades = evaluar_modelo(paquete_modelo,lista_feature_names1_sum,X_test_sum,Y_test_etiquetas,encoder_etiquetas)

    accuracy        = resultado['accuracy']
    macro_f1        = resultado['f1']
    macro_precision = resultado['precision']
    macro_recall    = resultado['recall']
    


    
    if macro_f1 >= mejor_macro_f1_global:
        mejor_macro_f1_global = macro_f1
        mejor_features_global = paquete_modelo['lista_features'].copy()
        pasos_sin_mejora = 0  # ¡Se reinicia porque hemos batido o empatado el récord global!
        mejor_paquete_modelo=paquete_modelo.copy()
    else:
        pasos_sin_mejora += 1  # Si es peor que el récord histórico, sumamos 1
        
    # Imprimir con el formato solicitado (mostrando ya el contador de "Sin mejora" actualizado)
    print(f"Model = {model_name:40s} Accuracy = {accuracy:.4f} | Macro F1 = {macro_f1:.4f} | Macro Precision = {macro_precision:.4f} | Macro Recall = {macro_recall:.4f} | Sin mejora: {pasos_sin_mejora}")
        

print(f"\n¡Proceso finalizado!")
print(f"Mejor Macro F1 alcanzado: {mejor_macro_f1_global:.4f}")
print(f"Número óptimo de características base: {len(mejor_features_global)}")

Iniciando eliminación recursiva desde 78 características.

Model = RandomForest (metricas_logs 1 punto suma)   N_feat=72 Accuracy = 0.6093 | Macro F1 = 0.6001 | Macro Precision = 0.6090 | Macro Recall = 0.6093 | Sin mejora: 1
Model = RandomForest (metricas_logs 1 punto suma)   N_feat=71 Accuracy = 0.6055 | Macro F1 = 0.5986 | Macro Precision = 0.6087 | Macro Recall = 0.6055 | Sin mejora: 2
Model = RandomForest (metricas_logs 1 punto suma)   N_feat=70 Accuracy = 0.6035 | Macro F1 = 0.5960 | Macro Precision = 0.6033 | Macro Recall = 0.6035 | Sin mejora: 3
Model = RandomForest (metricas_logs 1 punto suma)   N_feat=69 Accuracy = 0.6032 | Macro F1 = 0.5941 | Macro Precision = 0.6017 | Macro Recall = 0.6032 | Sin mejora: 4
Model = RandomForest (metricas_logs 1 punto suma)   N_feat=68 Accuracy = 0.5579 | Macro F1 = 0.5460 | Macro Precision = 0.5544 | Macro Recall = 0.5579 | Sin mejora: 5
Model = RandomForest (metricas_logs 1 punto suma)   N_feat=67 Accuracy = 0.5417 | Macro F1 = 0.5305 | Macr

In [109]:
mejor_paquete_modelo['model_name']

'RandomForest (metricas_logs 1 punto suma)   N_feat=78'

In [110]:
joblib.dump(mejor_paquete_modelo, f"{FOLDER_SALIDA}/models/{mejor_paquete_modelo['model_name']}.joblib")

['result/models/RandomForest (metricas_logs 1 punto suma)   N_feat=78.joblib']

In [117]:
mejor_paquete_modelo['resultado']

{'accuracy': 0.6275761973875181,
 'macro_f1': 0.6197418658287879,
 'macro_precision': 0.6283339292003264,
 'macro_recall': 0.6275761973875182}

## <span style="color:#2ca02c">4.2. Clasificador XGBoost con metricas_logs</span>

In [34]:
ALGORITMO_CLASIFICADOR = XGBClassifier
nombre_algoritmo       ='XGBoost'

## <span style="color:#2ca02c">4.2.1. Clasificador XGBoost (3 puntos)con metricas_logs</span>

In [35]:
N_puntos = 3
DATOS    = 'metricas_logs 3 puntos'

In [43]:
#hacemos esto para preparar los datos en las mismas variables todos los algoritmos

lista_features = lista_features_names_3puntos

X_train = X_matrix_sel_data(X_matrix,lista_feature_names,lista_idx_filas=idx_train,lista_feature_sel=lista_features)
print(X_train.shape)
X_test  = X_matrix_sel_data(X_matrix,lista_feature_names,lista_idx_filas=idx_test ,lista_feature_sel=lista_features)
print(X_test.shape)
X_val   = X_matrix_sel_data(X_matrix,lista_feature_names,lista_idx_filas=idx_val ,lista_feature_sel=lista_features)
print(X_val.shape)


(427, 993)
(143, 993)
(143, 993)


In [44]:
#la funcion objetivo y el conjunto de parametros es especifico del algoritmo por eso los dejamos en una celda aparte

params_grid = {
    'n_estimators': [200, 300, 400],
    'learning_rate': [0.01, 0.05, 0.1],
    'max_depth': [6, 8, 10],
    'subsample': [0.7, 0.8, 0.9, 1.0],
    'colsample_bytree': [0.7, 0.8, 0.9, 1.0],
    'min_child_weight': [1, 2, 3],
    'gamma': [0.0, 0.1, 0.2]
}





In [91]:
optuna.logging.set_verbosity(optuna.logging.WARNING)
N_trials = 30

# Barra de progreso global
pbar = tqdm(total=N_trials, desc=f"Optimizando {nombre_algoritmo}")

# Crear el estudio

study = optuna.create_study(
    direction='maximize',
    sampler=TPESampler(seed=SEED) 
)

# Enlazamos DIRECTAMENTE 'objective_universal' en el partial (evita usar alias antiguos)
objective_func = partial(
    objective_universal, 
    config=params_grid,
    clasificador_cls=ALGORITMO_CLASIFICADOR,
    X_train=X_train,
    X_test=X_test,
    Y_train_encoded=Y_train_encoded,
    Y_test_encoded=Y_test_encoded
)

# Ejecutar la optimización
print(f"=== INICIANDO OPTIMIZACIÓN DE {nombre_algoritmo} CON OPTUNA ===")
try:
    study.optimize(objective_func, n_trials=N_trials, callbacks=[print_callback], n_jobs=1)
finally:
    pbar.close()

# Resultados finales
print(f"\n=== MEJOR COMBINACIÓN ENCONTRADA ({nombre_algoritmo}) ===")
print(study.best_params)
print(f"Mejor Accuracy: {study.best_value:.4f}")

# Reconstruir el modelo final con los mejores parámetros óptimos
mejores_params = study.best_params.copy()
mejores_params['random_state'] = SEED
if 'n_jobs' in ALGORITMO_CLASIFICADOR().get_params():
    mejores_params['n_jobs'] = -1

Optimizando XGBoost:   0%|          | 0/30 [00:00<?, ?it/s]

=== INICIANDO OPTIMIZACIÓN DE XGBoost CON OPTUNA ===


Optimizando XGBoost:   3%|▎         | 1/30 [02:43<1:19:07, 163.71s/it, Mejor_Acc=0.6578, Trial=0]

Trial  0                | Actual Acc: 0.6578 | Parámetros: {'n_estimators': 200, 'learning_rate': 0.01, 'max_depth': 8, 'subsample': 0.8, 'colsample_bytree': 0.9, 'min_child_weight': 2, 'gamma': 0.1}
   -> MEJOR HASTA AHORA | Mejor Acc:  0.6578 | Parámetros: {'n_estimators': 200, 'learning_rate': 0.01, 'max_depth': 8, 'subsample': 0.8, 'colsample_bytree': 0.9, 'min_child_weight': 2, 'gamma': 0.1}
----------------------------------------------------------------------------------------------------


Optimizando XGBoost:   7%|▋         | 2/30 [05:00<1:09:06, 148.07s/it, Mejor_Acc=0.6763, Trial=1]

...

Trial 27                | Actual Acc: 0.6882 | Parámetros: {'n_estimators': 200, 'learning_rate': 0.05, 'max_depth': 6, 'subsample': 0.8, 'colsample_bytree': 1.0, 'min_child_weight': 3, 'gamma': 0.2}
   -> MEJOR HASTA AHORA | Mejor Acc:  0.6882 | Parámetros: {'n_estimators': 200, 'learning_rate': 0.05, 'max_depth': 6, 'subsample': 0.8, 'colsample_bytree': 1.0, 'min_child_weight': 3, 'gamma': 0.2}
----------------------------------------------------------------------------------------------------


Optimizando XGBoost:  97%|█████████▋| 29/30 [1:37:06<02:13, 133.37s/it, Mejor_Acc=0.6882, Trial=28]

Trial 28                | Actual Acc: 0.6557 | Parámetros: {'n_estimators': 200, 'learning_rate': 0.01, 'max_depth': 6, 'subsample': 0.8, 'colsample_bytree': 1.0, 'min_child_weight': 3, 'gamma': 0.2}
   -> MEJOR HASTA AHORA | Mejor Acc:  0.6882 | Parámetros: {'n_estimators': 200, 'learning_rate': 0.05, 'max_depth': 6, 'subsample': 0.8, 'colsample_bytree': 1.0, 'min_child_weight': 3, 'gamma': 0.2}
----------------------------------------------------------------------------------------------------


Optimizando XGBoost: 100%|██████████| 30/30 [1:39:09<00:00, 198.32s/it, Mejor_Acc=0.6882, Trial=29]

Trial 29                | Actual Acc: 0.6795 | Parámetros: {'n_estimators': 200, 'learning_rate': 0.1, 'max_depth': 6, 'subsample': 0.8, 'colsample_bytree': 1.0, 'min_child_weight': 3, 'gamma': 0.1}
   -> MEJOR HASTA AHORA | Mejor Acc:  0.6882 | Parámetros: {'n_estimators': 200, 'learning_rate': 0.05, 'max_depth': 6, 'subsample': 0.8, 'colsample_bytree': 1.0, 'min_child_weight': 3, 'gamma': 0.2}
----------------------------------------------------------------------------------------------------

=== MEJOR COMBINACIÓN ENCONTRADA (XGBoost) ===
{'n_estimators': 200, 'learning_rate': 0.05, 'max_depth': 6, 'subsample': 0.8, 'colsample_bytree': 1.0, 'min_child_weight': 3, 'gamma': 0.2}
Mejor Accuracy: 0.6882


In [145]:
#mejores_params = {'n_estimators': 200, 'learning_rate': 0.05, 'max_depth': 6, 'subsample': 0.8, 'colsample_bytree': 1.0, 'min_child_weight': 3, 'gamma': 0.2,'random_state': 100,
 'n_jobs': -1}

In [146]:
clf_anomalias_opt = ALGORITMO_CLASIFICADOR(**mejores_params)
clf_anomalias_opt.fit(X_train, Y_train_encoded)

,"objective objective: typing.Union[str, xgboost.sklearn._SklObjWProto, typing.Callable[[typing.Any, typing.Any], typing.Tuple[numpy.ndarray, numpy.ndarray]], NoneType]Specify the learning task and the corresponding learning objective or a customobjective function to be used.For custom objective, see :doc:`/tutorials/custom_metric_obj` and:ref:`custom-obj-metric` for more information, along with the end note forfunction signatures.",'multi:softprob'
,"base_score base_score: typing.Union[float, typing.List[float], NoneType]The initial prediction score of all instances, global bias.",None
,booster,None
,"callbacks callbacks: typing.Optional[typing.List[xgboost.callback.TrainingCallback]]List of callback functions that are applied at end of each iteration.It is possible to use predefined callbacks by using:ref:`Callback API <callback_api>`... note:: States in callback are not preserved during training, which means callback objects can not be reused for multiple training sessions without reinitialization or deepcopy... code-block:: python for params in parameters_grid: # be sure to (re)initialize the callbacks before each run callbacks = [xgb.callback.LearningRateScheduler(custom_rates)] reg = xgboost.XGBRegressor(**params, callbacks=callbacks) reg.fit(X, y)",None
,colsample_bylevel colsample_bylevel: typing.Optional[float]Subsample ratio of columns for each level.,None
,colsample_bynode colsample_bynode: typing.Optional[float]Subsample ratio of columns for each split.,None
,colsample_bytree colsample_bytree: typing.Optional[float]Subsample ratio of columns when constructing each tree.,1.0
,"device device: typing.Optional[str].. versionadded:: 2.0.0Device ordinal, available options are `cpu`, `cuda`, and `gpu`.",None
,"early_stopping_rounds early_stopping_rounds: typing.Optional[int].. versionadded:: 1.6.0- Activates early stopping. Validation metric needs to improve at least once in every **early_stopping_rounds** round(s) to continue training. Requires at least one item in **eval_set** in :py:meth:`fit`.- If early stopping occurs, the model will have two additional attributes: :py:attr:`best_score` and :py:attr:`best_iteration`. These are used by the :py:meth:`predict` and :py:meth:`apply` methods to determine the optimal number of trees during inference. If users want to access the full model (including trees built after early stopping), they can specify the `iteration_range` in these inference methods. In addition, other utilities like model plotting can also use the entire model.- If you prefer to discard the trees after `best_iteration`, consider using the callback function :py:class:`xgboost.callback.EarlyStopping`.- If there's more than one item in **eval_set**, the last entry will be used for early stopping. If there's more than one metric in **eval_metric**, the last metric will be used for early stopping.",None
,enable_categorical enable_categorical: boolSee the same parameter of :py:class:`DMatrix` for details.,False
,"eval_metric eval_metric: typing.Union[str, typing.List[typing.Union[str, typing.Callable]], typing.Callable, NoneType].. versionadded:: 1.6.0Metric used for monitoring the training result and early stopping. It can be astring or list of strings as names of predefined metric in XGBoost (See:doc:`/parameter`), one of the metrics in :py:mod:`sklearn.metrics`, or anyother user defined metric that looks like `sklearn.metrics`.If custom objective is also provided, then custom metric should implement thecorresponding reverse link function.Unlike the `scoring` parameter commonly used in scikit-learn, when a callableobject is provided, it's assumed to be a cost function and by default XGBoostwill minimize the result during early stopping.For advanced usage on Early stopping like directly choosing to maximize insteadof minimize, see :py:obj:`xgboost.callback.EarlyStopping`.See :doc:`/tutorials/custom_metric_obj` and :ref:`custom-obj-metric` for moreinformation... code-block:: python from sklearn.datasets import lo

In [147]:

model_name=f'{nombre_algoritmo} ({DATOS})(todas las features)'
paquete_modelo={}

paquete_modelo['model_name']         = model_name
paquete_modelo['model']              = clf_anomalias_opt
paquete_modelo['parametros']         = mejores_params
paquete_modelo['lista_features']     = lista_features
paquete_modelo['N_features']         = len(paquete_modelo['lista_features'] )
paquete_modelo['N_features_unicas']  = len(get_lista_feature_sin_lag(paquete_modelo['lista_features'] ))
paquete_modelo['datos']              = DATOS
paquete_modelo['algoritmo']          = nombre_algoritmo

resultado,reporte_dict,Y_pred_probabilidades = evaluar_modelo(paquete_modelo,lista_features,X_test,Y_test_etiquetas,encoder_etiquetas)
print(f"=== REPORTE DE CLASIFICACIÓN DE TIPOS DE ANOMALÍA ===\n{model_name}")
print(paquete_modelo['classification_report_str'])

joblib.dump(paquete_modelo, f'{FOLDER_SALIDA}/models/{model_name}.joblib')

=== REPORTE DE CLASIFICACIÓN DE TIPOS DE ANOMALÍA ===
XGBoost (metricas_logs 3 puntos)(todas las features)
                                               precision    recall  f1-score   support

           access_token_auth_header_error_401       1.00      0.98      0.99        65
          access_token_authorization_form_401       0.48      0.60      0.53        65
         access_token_client_id_not_found_404       0.98      1.00      0.99        65
         access_token_client_secret_wrong_401       0.98      0.97      0.98        65
             access_token_form_urlencoded_400       0.65      0.60      0.62        65
          access_token_illegal_grant_type_400       1.00      1.00      1.00        65
access_token_missing_authorization_header_400       0.54      0.54      0.54        65
     authorization_code_client_id_missing_400       0.81      0.77      0.79        65
     authorization_code_invalid_client_id_404       0.79      0.85      0.81        65
      authorization_co

['result/models/XGBoost (metricas_logs 3 puntos)(todas las features).joblib']

In [148]:
df_importancias_agrupada, importancias, num_cero = obtener_importancias_agrupadas(paquete_modelo['model'].feature_importances_, paquete_modelo['lista_features'])
print(f"Características con importancia cero: {num_cero}")

Características con importancia cero: 77


In [165]:
n_min = 54
n_max = len(importancias) - num_cero

lista_result = []

# --- VARIABLES PARA EL EARLY STOPPING ---
paciencia = 20
iteraciones_sin_mejora = 0
mejor_score_global = -1.0
mejor_paquete_modelo = None

for N_feat in range(n_min, n_max + 1):
    model_name = f'{nombre_algoritmo} ({DATOS})   N_feat={N_feat}'
    lista_features_sel = []
    for feature in df_importancias_agrupada['caracteristica'].to_list()[:N_feat]:
        if N_puntos>1:
            for lag in range(0, N_puntos):
                lista_features_sel.append(f'{feature}(lag_{lag})')
        else:
            lista_features_sel.append(feature)

    X_train_sub = X_matrix_sel_data(X_matrix, lista_feature_names, lista_idx_filas=idx_train, lista_feature_sel=lista_features_sel)
    X_test_sub  = X_matrix_sel_data(X_matrix, lista_feature_names, lista_idx_filas=idx_test,  lista_feature_sel=lista_features_sel)
   
    clf_anomalias_opt = ALGORITMO_CLASIFICADOR(**mejores_params)
    clf_anomalias_opt.fit(X_train_sub, Y_train_encoded)

    paquete_modelo = {}
    paquete_modelo['model_name']          = model_name
    paquete_modelo['model']               = clf_anomalias_opt
    paquete_modelo['parametros']          = mejores_params
    paquete_modelo['lista_features']      = lista_features_sel
    paquete_modelo['N_features']          = len(paquete_modelo['lista_features'])
    paquete_modelo['N_features_unicas']   = len(get_lista_feature_sin_lag(paquete_modelo['lista_features']))
    paquete_modelo['datos']               = 'metricas_log_lags'
    paquete_modelo['algoritmo']           = nombre_algoritmo
    
    resultado, reporte_dict, Y_pred_probabilidades = evaluar_modelo(paquete_modelo, lista_features, X_test, Y_test_etiquetas, encoder_etiquetas)
    accuracy        = resultado['accuracy']
    macro_f1        = resultado['f1']
    macro_precision = resultado['precision']
    macro_recall    = resultado['recall']
    
    # paquete_modelo['metrics'] = {
    #     'accuracy': accuracy,
    #     'macro_f1': macro_f1,
    #     'macro_precision': macro_precision,
    #     'macro_recall': macro_recall
    # }
    
    lista_result.append(paquete_modelo)

    # --- LÓGICA DE EARLY STOPPING ---
    score_actual = macro_f1 
    
    if score_actual > mejor_score_global:
        mejor_score_global = score_actual
        mejor_paquete_modelo = paquete_modelo.copy()
        iteraciones_sin_mejora = 0  # Reiniciamos contador si mejora
    else:
        iteraciones_sin_mejora += 1

    # --- IMPRESIÓN EN UNA SOLA LÍNEA ---
    print(f"Model = {model_name:40s} Accuracy = {accuracy:.4f} | Macro F1 = {macro_f1:.4f} | Macro Precision = {macro_precision:.4f} | Macro Recall = {macro_recall:.4f} | Mejor Macro F1 = {mejor_score_global:.4f} | Iteraciones_sin_mejora = {iteraciones_sin_mejora}")

    # Comprobación de parada
    if iteraciones_sin_mejora >= paciencia:
        print(f"\n[!] Early Stopping activado: No ha habido mejoras en {paciencia} iteraciones consecutivas.")
        print(f"Deteniendo búsqueda. El mejor modelo se logró con {mejor_paquete_modelo['N_features_unicas']} características base.")
        break

Model = XGBoost (metricas_logs 3 puntos)   N_feat=54 Accuracy = 0.6888 | Macro F1 = 0.6878 | Macro Precision = 0.6901 | Macro Recall = 0.6888 | Mejor Macro F1 = 0.6878 | Iteraciones_sin_mejora = 0
Model = XGBoost (metricas_logs 3 puntos)   N_feat=55 Accuracy = 0.6891 | Macro F1 = 0.6878 | Macro Precision = 0.6906 | Macro Recall = 0.6891 | Mejor Macro F1 = 0.6878 | Iteraciones_sin_mejora = 1
Model = XGBoost (metricas_logs 3 puntos)   N_feat=56 Accuracy = 0.6894 | Macro F1 = 0.6876 | Macro Precision = 0.6913 | Macro Recall = 0.6894 | Mejor Macro F1 = 0.6878 | Iteraciones_sin_mejora = 2
Model = XGBoost (metricas_logs 3 puntos)   N_feat=57 Accuracy = 0.6871 | Macro F1 = 0.6852 | Macro Precision = 0.6892 | Macro Recall = 0.6871 | Mejor Macro F1 = 0.6878 | Iteraciones_sin_mejora = 3
Model = XGBoost (metricas_logs 3 puntos)   N_feat=58 Accuracy = 0.6833 | Macro F1 = 0.6818 | Macro Precision = 0.6847 | Macro Recall = 0.6833 | Mejor Macro F1 = 0.6878 | Iteraciones_sin_mejora = 4
Model = XGBoost

In [166]:
mejor_paquete_modelo['model_name']

'XGBoost (metricas_logs 3 puntos)   N_feat=102'

In [167]:
joblib.dump(mejor_paquete_modelo, f"{FOLDER_SALIDA}/models/{mejor_paquete_modelo['model_name']}.joblib")

['result/models/XGBoost (metricas_logs 3 puntos)   N_feat=102.joblib']

In [168]:
paquete_modelo = mejor_paquete_modelo.copy()

mejor_macro_f1_global    = paquete_modelo['resultado']['f1']
mejor_features_global    = paquete_modelo['lista_features'].copy()
pasos_sin_mejora         = 0
limite_puntos_sin_mejora = paciencia + 5

print(f"Iniciando eliminación recursiva desde {paquete_modelo['N_features_unicas']} características.\n")

while paquete_modelo['N_features_unicas'] > 1 and pasos_sin_mejora < limite_puntos_sin_mejora:
    N_feat = paquete_modelo['N_features_unicas']
    
    # Extraer importancias y características actuales
    importancias_raw = paquete_modelo['model'].feature_importances_
    features_raw = paquete_modelo['lista_features']
    
    df_importancias_agrupada, _, num_cero = obtener_importancias_agrupadas(importancias_raw, features_raw)

    # Seleccionar características para el siguiente paso
    lista_features_sel = []
    for feature in df_importancias_agrupada['caracteristica'].to_list()[:(N_feat-1)]:
        if N_puntos>1:
            for lag in range(0, N_puntos):
                lista_features_sel.append(f'{feature}(lag_{lag})')
        else:
            lista_features_sel.append(f'{feature}')
            
    X_train_sub = X_matrix_sel_data(X_matrix, lista_feature_names, lista_idx_filas=idx_train, lista_feature_sel=lista_features_sel)
    X_test_sub  = X_matrix_sel_data(X_matrix, lista_feature_names, lista_idx_filas=idx_test,  lista_feature_sel=lista_features_sel)
   
    clf_anomalias_opt = ALGORITMO_CLASIFICADOR(**mejores_params)
    clf_anomalias_opt.fit(X_train_sub, Y_train_encoded)

    model_name = f'{nombre_algoritmo} (metricas_log) N_feat={N_feat-1}'
    paquete_modelo = {}
    paquete_modelo['model_name']          = model_name
    paquete_modelo['model']               = clf_anomalias_opt
    paquete_modelo['parametros']          = mejores_params
    paquete_modelo['lista_features']      = lista_features_sel
    paquete_modelo['N_features']          = len(paquete_modelo['lista_features'])
    paquete_modelo['N_features_unicas']   = len(get_lista_feature_sin_lag(paquete_modelo['lista_features']))
    paquete_modelo['datos']               = 'metricas_log_lags'
    paquete_modelo['algoritmo']           = nombre_algoritmo
    
    resultado, reporte_dict, Y_pred_probabilidades = evaluar_modelo(paquete_modelo, lista_features, X_test, Y_test_etiquetas, encoder_etiquetas)
   
    accuracy        = resultado['accuracy']
    macro_f1        = resultado['f1']
    macro_precision = resultado['precision']
    macro_recall    = resultado['recall']
    

    
    if macro_f1 >= mejor_macro_f1_global:
        mejor_macro_f1_global = macro_f1
        mejor_features_global = paquete_modelo['lista_features'].copy()
        pasos_sin_mejora = 0  
        mejor_paquete_modelo = paquete_modelo.copy()
    else:
        pasos_sin_mejora += 1  
        
    print(f"Model = {model_name:40s} Accuracy = {accuracy:.4f} | Macro F1 = {macro_f1:.4f} | Macro Precision = {macro_precision:.4f} | Macro Recall = {macro_recall:.4f} | Sin mejora: {pasos_sin_mejora}")
    del X_train_sub, X_test_sub, importancias_raw, df_importancias_agrupada
    gc.collect()

print(f"\n¡Proceso finalizado!")
print(f"Mejor Macro F1 alcanzado: {mejor_macro_f1_global:.4f}")
print(f"Número óptimo de características base: {len(mejor_features_global)}")


Iniciando eliminación recursiva desde 102 características.

Model = XGBoost (metricas_log) N_feat=101        Accuracy = 0.6888 | Macro F1 = 0.6875 | Macro Precision = 0.6893 | Macro Recall = 0.6888 | Sin mejora: 1
Model = XGBoost (metricas_log) N_feat=100        Accuracy = 0.6348 | Macro F1 = 0.6316 | Macro Precision = 0.6329 | Macro Recall = 0.6348 | Sin mejora: 2
Model = XGBoost (metricas_log) N_feat=99         Accuracy = 0.6377 | Macro F1 = 0.6346 | Macro Precision = 0.6367 | Macro Recall = 0.6377 | Sin mejora: 3
Model = XGBoost (metricas_log) N_feat=98         Accuracy = 0.6313 | Macro F1 = 0.6273 | Macro Precision = 0.6281 | Macro Recall = 0.6313 | Sin mejora: 4
Model = XGBoost (metricas_log) N_feat=97         Accuracy = 0.6343 | Macro F1 = 0.6310 | Macro Precision = 0.6324 | Macro Recall = 0.6343 | Sin mejora: 5
Model = XGBoost (metricas_log) N_feat=96         Accuracy = 0.6308 | Macro F1 = 0.6266 | Macro Precision = 0.6270 | Macro Recall = 0.6308 | Sin mejora: 6
Model = XGBoost 

In [169]:
mejor_paquete_modelo['model_name']

'XGBoost (metricas_logs 3 puntos)   N_feat=102'

In [170]:
joblib.dump(mejor_paquete_modelo, f"{FOLDER_SALIDA}/models/{mejor_paquete_modelo['model_name']}.joblib")

['result/models/XGBoost (metricas_logs 3 puntos)   N_feat=102.joblib']

### <span style="color:#2ca02c">4.5.2. Clasificador XGBoost (1 punto de 3 intervalos) con metricas_logs</span>

In [171]:
N_puntos = 1
DATOS    = 'metricas_logs 1 punto suma'

In [172]:
lista_features = lista_feature_names_1punto_sum

X_train = X_matrix_sel_data(X_matrix,lista_feature_names,lista_idx_filas=idx_train,lista_feature_sel=lista_features)
print(X_train.shape)
X_test  = X_matrix_sel_data(X_matrix,lista_feature_names,lista_idx_filas=idx_test ,lista_feature_sel=lista_features)
print(X_test.shape)
X_val   = X_matrix_sel_data(X_matrix,lista_feature_names,lista_idx_filas=idx_val ,lista_feature_sel=lista_features)
print(X_val.shape)

(10282, 331)
(3445, 331)
(3445, 331)


In [173]:
params_grid= {
    'n_estimators': [ 200, 300, 400],
    'learning_rate': [0.05, 0.01, 0.05],
    'max_depth': [6, 8, 10],
    'subsample': [0.7, 0.8,0.9, 1.0],
    'colsample_bytree': [0.7, 0.8, 0.9, 1.0],
    'min_child_weight': [1, 2,  3],
    'gamma': [0.0, 0.1, 0.2]
}

In [174]:
optuna.logging.set_verbosity(optuna.logging.WARNING)
N_trials = 30

# Barra de progreso global
pbar = tqdm(total=N_trials, desc=f"Optimizando {nombre_algoritmo}")

# Crear el estudio

study = optuna.create_study(
    direction='maximize',
    sampler=TPESampler(seed=SEED) 
)

# Enlazamos DIRECTAMENTE 'objective_universal' en el partial (evita usar alias antiguos)
objective_func = partial(
    objective_universal, 
    config=params_grid,
    clasificador_cls=ALGORITMO_CLASIFICADOR,
    X_train=X_train,
    X_test=X_test,
    Y_train_encoded=Y_train_encoded,
    Y_test_encoded=Y_test_encoded
)

# Ejecutar la optimización
print(f"=== INICIANDO OPTIMIZACIÓN DE {nombre_algoritmo} CON OPTUNA ===")
try:
    study.optimize(objective_func, n_trials=N_trials, callbacks=[print_callback], n_jobs=1)
finally:
    pbar.close()

# Resultados finales
print(f"\n=== MEJOR COMBINACIÓN ENCONTRADA ({nombre_algoritmo}) ===")
print(study.best_params)
print(f"Mejor Accuracy: {study.best_value:.4f}")

# Reconstruir el modelo final con los mejores parámetros óptimos
mejores_params = study.best_params.copy()
mejores_params['random_state'] = SEED
if 'n_jobs' in ALGORITMO_CLASIFICADOR().get_params():
    mejores_params['n_jobs'] = -1

Optimizando XGBoost:   0%|          | 0/30 [00:00<?, ?it/s]

=== INICIANDO OPTIMIZACIÓN DE XGBoost CON OPTUNA ===


Optimizando XGBoost:   3%|▎         | 1/30 [00:45<21:46, 45.04s/it, Mejor_Acc=0.6589, Trial=0]

Trial  0                | Actual Acc: 0.6589 | Parámetros: {'n_estimators': 200, 'learning_rate': 0.05, 'max_depth': 8, 'subsample': 0.8, 'colsample_bytree': 0.9, 'min_child_weight': 2, 'gamma': 0.1}
   -> MEJOR HASTA AHORA | Mejor Acc:  0.6589 | Parámetros: {'n_estimators': 200, 'learning_rate': 0.05, 'max_depth': 8, 'subsample': 0.8, 'colsample_bytree': 0.9, 'min_child_weight': 2, 'gamma': 0.1}
----------------------------------------------------------------------------------------------------


Optimizando XGBoost:   7%|▋         | 2/30 [01:44<24:53, 53.34s/it, Mejor_Acc=0.6589, Trial=1]

...

Trial 27                | Actual Acc: 0.6639 | Parámetros: {'n_estimators': 300, 'learning_rate': 0.05, 'max_depth': 6, 'subsample': 0.8, 'colsample_bytree': 1.0, 'min_child_weight': 3, 'gamma': 0.1}
   -> MEJOR HASTA AHORA | Mejor Acc:  0.6639 | Parámetros: {'n_estimators': 300, 'learning_rate': 0.05, 'max_depth': 6, 'subsample': 0.8, 'colsample_bytree': 1.0, 'min_child_weight': 3, 'gamma': 0.1}
----------------------------------------------------------------------------------------------------


Optimizando XGBoost:  97%|█████████▋| 29/30 [27:33<00:59, 59.50s/it, Mejor_Acc=0.6639, Trial=28]

Trial 28                | Actual Acc: 0.6639 | Parámetros: {'n_estimators': 300, 'learning_rate': 0.05, 'max_depth': 6, 'subsample': 0.8, 'colsample_bytree': 1.0, 'min_child_weight': 3, 'gamma': 0.1}
   -> MEJOR HASTA AHORA | Mejor Acc:  0.6639 | Parámetros: {'n_estimators': 300, 'learning_rate': 0.05, 'max_depth': 6, 'subsample': 0.8, 'colsample_bytree': 1.0, 'min_child_weight': 3, 'gamma': 0.1}
----------------------------------------------------------------------------------------------------


Optimizando XGBoost: 100%|██████████| 30/30 [28:39<00:00, 57.30s/it, Mejor_Acc=0.6662, Trial=29]

Trial 29                | Actual Acc: 0.6662 | Parámetros: {'n_estimators': 400, 'learning_rate': 0.05, 'max_depth': 6, 'subsample': 0.8, 'colsample_bytree': 1.0, 'min_child_weight': 3, 'gamma': 0.1}
   -> MEJOR HASTA AHORA | Mejor Acc:  0.6662 | Parámetros: {'n_estimators': 400, 'learning_rate': 0.05, 'max_depth': 6, 'subsample': 0.8, 'colsample_bytree': 1.0, 'min_child_weight': 3, 'gamma': 0.1}
----------------------------------------------------------------------------------------------------

=== MEJOR COMBINACIÓN ENCONTRADA (XGBoost) ===
{'n_estimators': 400, 'learning_rate': 0.05, 'max_depth': 6, 'subsample': 0.8, 'colsample_bytree': 1.0, 'min_child_weight': 3, 'gamma': 0.1}
Mejor Accuracy: 0.6662


In [175]:
clf_anomalias_opt = ALGORITMO_CLASIFICADOR(**mejores_params)
clf_anomalias_opt.fit(X_train, Y_train_encoded)

,"objective objective: typing.Union[str, xgboost.sklearn._SklObjWProto, typing.Callable[[typing.Any, typing.Any], typing.Tuple[numpy.ndarray, numpy.ndarray]], NoneType]Specify the learning task and the corresponding learning objective or a customobjective function to be used.For custom objective, see :doc:`/tutorials/custom_metric_obj` and:ref:`custom-obj-metric` for more information, along with the end note forfunction signatures.",'multi:softprob'
,"base_score base_score: typing.Union[float, typing.List[float], NoneType]The initial prediction score of all instances, global bias.",None
,booster,None
,"callbacks callbacks: typing.Optional[typing.List[xgboost.callback.TrainingCallback]]List of callback functions that are applied at end of each iteration.It is possible to use predefined callbacks by using:ref:`Callback API <callback_api>`... note:: States in callback are not preserved during training, which means callback objects can not be reused for multiple training sessions without reinitialization or deepcopy... code-block:: python for params in parameters_grid: # be sure to (re)initialize the callbacks before each run callbacks = [xgb.callback.LearningRateScheduler(custom_rates)] reg = xgboost.XGBRegressor(**params, callbacks=callbacks) reg.fit(X, y)",None
,colsample_bylevel colsample_bylevel: typing.Optional[float]Subsample ratio of columns for each level.,None
,colsample_bynode colsample_bynode: typing.Optional[float]Subsample ratio of columns for each split.,None
,colsample_bytree colsample_bytree: typing.Optional[float]Subsample ratio of columns when constructing each tree.,1.0
,"device device: typing.Optional[str].. versionadded:: 2.0.0Device ordinal, available options are `cpu`, `cuda`, and `gpu`.",None
,"early_stopping_rounds early_stopping_rounds: typing.Optional[int].. versionadded:: 1.6.0- Activates early stopping. Validation metric needs to improve at least once in every **early_stopping_rounds** round(s) to continue training. Requires at least one item in **eval_set** in :py:meth:`fit`.- If early stopping occurs, the model will have two additional attributes: :py:attr:`best_score` and :py:attr:`best_iteration`. These are used by the :py:meth:`predict` and :py:meth:`apply` methods to determine the optimal number of trees during inference. If users want to access the full model (including trees built after early stopping), they can specify the `iteration_range` in these inference methods. In addition, other utilities like model plotting can also use the entire model.- If you prefer to discard the trees after `best_iteration`, consider using the callback function :py:class:`xgboost.callback.EarlyStopping`.- If there's more than one item in **eval_set**, the last entry will be used for early stopping. If there's more than one metric in **eval_metric**, the last metric will be used for early stopping.",None
,enable_categorical enable_categorical: boolSee the same parameter of :py:class:`DMatrix` for details.,False
,"eval_metric eval_metric: typing.Union[str, typing.List[typing.Union[str, typing.Callable]], typing.Callable, NoneType].. versionadded:: 1.6.0Metric used for monitoring the training result and early stopping. It can be astring or list of strings as names of predefined metric in XGBoost (See:doc:`/parameter`), one of the metrics in :py:mod:`sklearn.metrics`, or anyother user defined metric that looks like `sklearn.metrics`.If custom objective is also provided, then custom metric should implement thecorresponding reverse link function.Unlike the `scoring` parameter commonly used in scikit-learn, when a callableobject is provided, it's assumed to be a cost function and by default XGBoostwill minimize the result during early stopping.For advanced usage on Early stopping like directly choosing to maximize insteadof minimize, see :py:obj:`xgboost.callback.EarlyStopping`.See :doc:`/tutorials/custom_metric_obj` and :ref:`custom-obj-metric` for moreinformation... code-block:: python from sklearn.datasets import lo

In [176]:
model_name=f'{nombre_algoritmo} ({DATOS})(todas las features)'
paquete_modelo={}

paquete_modelo['model_name']         = model_name
paquete_modelo['model']              = clf_anomalias_opt
paquete_modelo['parametros']         = mejores_params
paquete_modelo['lista_features']     = lista_features
paquete_modelo['N_features']         = len(paquete_modelo['lista_features'] )
paquete_modelo['N_features_unicas']  = len(get_lista_feature_sin_lag(paquete_modelo['lista_features'] ))
paquete_modelo['datos']              = DATOS
paquete_modelo['algoritmo']          = nombre_algoritmo

resultado,reporte_dict,Y_pred_probabilidades = evaluar_modelo(paquete_modelo,lista_features,X_test,Y_test_etiquetas,encoder_etiquetas)
print(f"=== REPORTE DE CLASIFICACIÓN DE TIPOS DE ANOMALÍA ===\n{model_name}")
print(paquete_modelo['classification_report_str'])

joblib.dump(paquete_modelo, f'{FOLDER_SALIDA}/models/{model_name}.joblib')


=== REPORTE DE CLASIFICACIÓN DE TIPOS DE ANOMALÍA ===
XGBoost (metricas_logs 1 punto suma)(todas las features)
                                               precision    recall  f1-score   support

           access_token_auth_header_error_401       0.94      0.97      0.95        65
          access_token_authorization_form_401       0.47      0.54      0.50        65
         access_token_client_id_not_found_404       0.98      1.00      0.99        65
         access_token_client_secret_wrong_401       0.98      0.97      0.98        65
             access_token_form_urlencoded_400       0.70      0.66      0.68        65
          access_token_illegal_grant_type_400       1.00      1.00      1.00        65
access_token_missing_authorization_header_400       0.49      0.54      0.51        65
     authorization_code_client_id_missing_400       0.86      0.83      0.84        65
     authorization_code_invalid_client_id_404       0.69      0.77      0.73        65
      authorizatio

['result/models/XGBoost (metricas_logs 1 punto suma)(todas las features).joblib']

In [177]:
df_importancias_agrupada, importancias, num_cero = obtener_importancias_agrupadas(paquete_modelo['model'].feature_importances_, paquete_modelo['lista_features'])
print(f"Características con importancia cero: {num_cero}")

Características con importancia cero: 69


In [180]:
n_min = 149
n_max = len(importancias) - num_cero


# --- VARIABLES PARA EL EARLY STOPPING ---
paciencia = 15
iteraciones_sin_mejora = 0
mejor_score_global = -1.0
mejor_paquete_modelo = None

for N_feat in range(n_min, n_max + 1):
    model_name = f'{nombre_algoritmo} ({DATOS})   N_feat={N_feat}'
    lista_features_sel = []
    for feature in df_importancias_agrupada['caracteristica'].to_list()[:N_feat]:
        if N_puntos>1:
            for lag in range(0, N_puntos):
                lista_features_sel.append(f'{feature}(lag_{lag})')
        else:
            lista_features_sel.append(feature)

    X_train_sub = X_matrix_sel_data(X_matrix, lista_feature_names, lista_idx_filas=idx_train, lista_feature_sel=lista_features_sel)
    X_test_sub  = X_matrix_sel_data(X_matrix, lista_feature_names, lista_idx_filas=idx_test,  lista_feature_sel=lista_features_sel)
   
    clf_anomalias_opt = ALGORITMO_CLASIFICADOR(**mejores_params)
    clf_anomalias_opt.fit(X_train_sub, Y_train_encoded)

    paquete_modelo = {}
    paquete_modelo['model_name']          = model_name
    paquete_modelo['model']               = clf_anomalias_opt
    paquete_modelo['parametros']          = mejores_params
    paquete_modelo['lista_features']      = lista_features_sel
    paquete_modelo['N_features']          = len(paquete_modelo['lista_features'])
    paquete_modelo['N_features_unicas']   = len(get_lista_feature_sin_lag(paquete_modelo['lista_features']))
    paquete_modelo['datos']               = 'metricas_log_lags'
    paquete_modelo['algoritmo']           = nombre_algoritmo
    
    resultado, reporte_dict, Y_pred_probabilidades = evaluar_modelo(paquete_modelo, lista_features, X_test, Y_test_etiquetas, encoder_etiquetas)
    accuracy        = resultado['accuracy']
    macro_f1        = resultado['f1']
    macro_precision = resultado['precision']
    macro_recall    = resultado['recall']
    


    # --- LÓGICA DE EARLY STOPPING ---
    score_actual = macro_f1 
    
    if score_actual > mejor_score_global:
        mejor_score_global = score_actual
        mejor_paquete_modelo = paquete_modelo.copy()
        iteraciones_sin_mejora = 0  # Reiniciamos contador si mejora
    else:
        iteraciones_sin_mejora += 1

    # --- IMPRESIÓN EN UNA SOLA LÍNEA ---
    print(f"Model = {model_name:40s} Accuracy = {accuracy:.4f} | Macro F1 = {macro_f1:.4f} | Macro Precision = {macro_precision:.4f} | Macro Recall = {macro_recall:.4f} | Mejor Macro F1 = {mejor_score_global:.4f} | Iteraciones_sin_mejora = {iteraciones_sin_mejora}")
    del X_train_sub, X_test_sub
    gc.collect()
    
    # Comprobación de parada
    if iteraciones_sin_mejora >= paciencia:
        print(f"\n[!] Early Stopping activado: No ha habido mejoras en {paciencia} iteraciones consecutivas.")
        print(f"Deteniendo búsqueda. El mejor modelo se logró con {mejor_paquete_modelo['N_features_unicas']} características base.")
        break


Model = XGBoost (metricas_logs 1 punto suma)   N_feat=149 Accuracy = 0.6543 | Macro F1 = 0.6509 | Macro Precision = 0.6530 | Macro Recall = 0.6543 | Mejor Macro F1 = 0.6509 | Iteraciones_sin_mejora = 0
Model = XGBoost (metricas_logs 1 punto suma)   N_feat=150 Accuracy = 0.6615 | Macro F1 = 0.6586 | Macro Precision = 0.6613 | Macro Recall = 0.6615 | Mejor Macro F1 = 0.6586 | Iteraciones_sin_mejora = 0
Model = XGBoost (metricas_logs 1 punto suma)   N_feat=151 Accuracy = 0.6604 | Macro F1 = 0.6571 | Macro Precision = 0.6594 | Macro Recall = 0.6604 | Mejor Macro F1 = 0.6586 | Iteraciones_sin_mejora = 1
Model = XGBoost (metricas_logs 1 punto suma)   N_feat=152 Accuracy = 0.6601 | Macro F1 = 0.6567 | Macro Precision = 0.6592 | Macro Recall = 0.6601 | Mejor Macro F1 = 0.6586 | Iteraciones_sin_mejora = 2
Model = XGBoost (metricas_logs 1 punto suma)   N_feat=153 Accuracy = 0.6592 | Macro F1 = 0.6562 | Macro Precision = 0.6586 | Macro Recall = 0.6592 | Mejor Macro F1 = 0.6586 | Iteraciones_sin_m

In [181]:
mejor_paquete_modelo['model_name']

'XGBoost (metricas_logs 1 punto suma)   N_feat=160'

In [182]:
joblib.dump(mejor_paquete_modelo, f"{FOLDER_SALIDA}/models/{mejor_paquete_modelo['model_name']}.joblib")

['result/models/XGBoost (metricas_logs 1 punto suma)   N_feat=160.joblib']

In [102]:
mejor_paquete_modelo = joblib.load('result/models/LightGBM (metricas_logs 1 punto suma)   N_feat=202.joblib')

In [103]:
paquete_modelo = mejor_paquete_modelo.copy()


# features_actuales = df_importancias_agrupada['caracteristica'].to_list()[:N_inicio]

mejor_macro_f1_global    = paquete_modelo['resultado']['f1']
mejor_features_global    = paquete_modelo['lista_features'].copy()
pasos_sin_mejora         = 0
limite_puntos_sin_mejora = 20

print(f"Iniciando eliminación recursiva desde {paquete_modelo['N_features_unicas']} características.\n")

while paquete_modelo['N_features_unicas'] > 1 and pasos_sin_mejora < limite_puntos_sin_mejora:

    df_importancias_agrupada, importancias, num_cero = obtener_importancias_agrupadas(paquete_modelo['model'].feature_importances_, paquete_modelo['lista_features'])
 
    lista_features_sel=[]
    for feature in df_importancias_agrupada['caracteristica'].to_list()[:(N_feat-1)]:
        if N_puntos>1:
            for lag in range(0, N_puntos):
                lista_features_sel.append(f'{feature}(lag_{lag})')
        else:
            lista_features_sel.append(f'{feature}')
    
    X_train_sub = X_matrix_sel_data(X_matrix, lista_feature_names, lista_idx_filas=idx_train, lista_feature_sel=lista_features_sel)
    X_test_sub  = X_matrix_sel_data(X_matrix, lista_feature_names, lista_idx_filas=idx_test,  lista_feature_sel=lista_features_sel)
   
    clf_anomalias_opt = ALGORITMO_CLASIFICADOR(**mejores_params)
    clf_anomalias_opt.fit(X_train_sub, Y_train_encoded)
    
    N_feat = len(lista_features_sel)
    model_name = f'{nombre_algoritmo} ({DATOS})   N_feat={N_feat}'

    paquete_modelo = {}
    paquete_modelo['model_name']          = model_name
    paquete_modelo['model']               = clf_anomalias_opt
    paquete_modelo['parametros']          = mejores_params
    paquete_modelo['lista_features']      = lista_features_sel
    paquete_modelo['N_features']          = len(paquete_modelo['lista_features'])
    paquete_modelo['N_features_unicas']   = len(get_lista_feature_sin_lag(paquete_modelo['lista_features']))
    paquete_modelo['datos']               = 'metricas_log_lags'
    paquete_modelo['algoritmo']           = nombre_algoritmo
    

    resultado, reporte_dict, Y_pred_probabilidades = evaluar_modelo(paquete_modelo,lista_feature_names1_sum,X_test_sum,Y_test_etiquetas,encoder_etiquetas)

    accuracy        = resultado['accuracy']
    macro_f1        = resultado['f1']
    macro_precision = resultado['precision']
    macro_recall    = resultado['recall']
    


    
    if macro_f1 >= mejor_macro_f1_global:
        mejor_macro_f1_global = macro_f1
        mejor_features_global = paquete_modelo['lista_features'].copy()
        pasos_sin_mejora = 0  # ¡Se reinicia porque hemos batido o empatado el récord global!
        mejor_paquete_modelo=paquete_modelo.copy()
    else:
        pasos_sin_mejora += 1  # Si es peor que el récord histórico, sumamos 1
        
    # Imprimir con el formato solicitado (mostrando ya el contador de "Sin mejora" actualizado)
    print(f"Model = {model_name:40s} Accuracy = {accuracy:.4f} | Macro F1 = {macro_f1:.4f} | Macro Precision = {macro_precision:.4f} | Macro Recall = {macro_recall:.4f} | Sin mejora: {pasos_sin_mejora}")
        

print(f"\n¡Proceso finalizado!")
print(f"Mejor Macro F1 alcanzado: {mejor_macro_f1_global:.4f}")
print(f"Número óptimo de características base: {len(mejor_features_global)}")

Iniciando eliminación recursiva desde 202 características.

Model = LightGBM (metricas_logs 1 punto suma)   N_feat=182 Accuracy = 0.6493 | Macro F1 = 0.6469 | Macro Precision = 0.6497 | Macro Recall = 0.6493 | Sin mejora: 1
Model = LightGBM (metricas_logs 1 punto suma)   N_feat=181 Accuracy = 0.6488 | Macro F1 = 0.6451 | Macro Precision = 0.6478 | Macro Recall = 0.6488 | Sin mejora: 2


KeyboardInterrupt: 

In [187]:
mejor_paquete_modelo['model_name']

'XGBoost (metricas_logs 1 punto suma)   N_feat=160'

In [ ]:
joblib.dump(mejor_paquete_modelo, f"{FOLDER_SALIDA}/models/{mejor_paquete_modelo['model_name']}.joblib")

In [59]:
paquete_modelo = mejor_paquete_modelo.copy()

mejor_macro_f1_global    = paquete_modelo['resultado']['f1']
mejor_features_global    = paquete_modelo['lista_features'].copy()
pasos_sin_mejora         = 0
limite_puntos_sin_mejora = paciencia + 5

print(f"Iniciando eliminación recursiva desde {paquete_modelo['N_features_unicas']} características.\n")

while paquete_modelo['N_features_unicas'] > 1 and pasos_sin_mejora < limite_puntos_sin_mejora:
    N_feat = paquete_modelo['N_features_unicas']
    
    # Extraer importancias y características actuales
    importancias_raw = paquete_modelo['model'].feature_importances_
    features_raw = paquete_modelo['lista_features']
    
    df_importancias_agrupada, _, num_cero = obtener_importancias_agrupadas(importancias_raw, features_raw)

    # Seleccionar características para el siguiente paso
    lista_features_sel = []
    for feature in df_importancias_agrupada['caracteristica'].to_list()[:(N_feat-1)]:
        if N_puntos>1:
            for lag in range(0, N_puntos):
                lista_features_sel.append(f'{feature}(lag_{lag})')
        else:
            lista_features_sel.append(f'{feature}')
    
    X_train_sub = X_matrix_sel_data(X_matrix, lista_feature_names, lista_idx_filas=idx_train, lista_feature_sel=lista_features_sel)
    X_test_sub  = X_matrix_sel_data(X_matrix, lista_feature_names, lista_idx_filas=idx_test,  lista_feature_sel=lista_features_sel)
   
    clf_anomalias_opt = ALGORITMO_CLASIFICADOR(**mejores_params)
    clf_anomalias_opt.fit(X_train_sub, Y_train_encoded)

    model_name = f'{nombre_algoritmo} (metricas_log) N_feat={N_feat-1}'
    paquete_modelo = {}
    paquete_modelo['model_name']          = model_name
    paquete_modelo['model']               = clf_anomalias_opt
    paquete_modelo['parametros']          = mejores_params
    paquete_modelo['lista_features']      = lista_features_sel
    paquete_modelo['N_features']          = len(paquete_modelo['lista_features'])
    paquete_modelo['N_features_unicas']   = len(get_lista_feature_sin_lag(paquete_modelo['lista_features']))
    paquete_modelo['datos']               = 'metricas_log_lags'
    paquete_modelo['algoritmo']           = nombre_algoritmo
    
    resultado, reporte_dict, Y_pred_probabilidades = evaluar_modelo(paquete_modelo, lista_features, X_test, Y_test_etiquetas, encoder_etiquetas)
   
    accuracy        = resultado['accuracy']
    macro_f1        = resultado['f1']
    macro_precision = resultado['precision']
    macro_recall    = resultado['recall']
    

    
    if macro_f1 >= mejor_macro_f1_global:
        mejor_macro_f1_global = macro_f1
        mejor_features_global = paquete_modelo['lista_features'].copy()
        pasos_sin_mejora = 0  
        mejor_paquete_modelo = paquete_modelo.copy()
    else:
        pasos_sin_mejora += 1  
        
    print(f"Model = {model_name:40s} Accuracy = {accuracy:.4f} | Macro F1 = {macro_f1:.4f} | Macro Precision = {macro_precision:.4f} | Macro Recall = {macro_recall:.4f} | Sin mejora: {pasos_sin_mejora}")
    del X_train_sub, X_test_sub, importancias_raw, df_importancias_agrupada
    gc.collect()

print(f"\n¡Proceso finalizado!")
print(f"Mejor Macro F1 alcanzado: {mejor_macro_f1_global:.4f}")
print(f"Número óptimo de características base: {len(mejor_features_global)}")


Iniciando eliminación recursiva desde 147 características.

Model = LightGBM (metricas_log) N_feat=146       Accuracy = 0.6958 | Macro F1 = 0.6950 | Macro Precision = 0.6972 | Macro Recall = 0.6958 | Sin mejora: 1
Model = LightGBM (metricas_log) N_feat=145       Accuracy = 0.6958 | Macro F1 = 0.6934 | Macro Precision = 0.6940 | Macro Recall = 0.6958 | Sin mejora: 2
Model = LightGBM (metricas_log) N_feat=144       Accuracy = 0.6926 | Macro F1 = 0.6919 | Macro Precision = 0.6943 | Macro Recall = 0.6926 | Sin mejora: 3
Model = LightGBM (metricas_log) N_feat=143       Accuracy = 0.6958 | Macro F1 = 0.6944 | Macro Precision = 0.6964 | Macro Recall = 0.6958 | Sin mejora: 4
Model = LightGBM (metricas_log) N_feat=142       Accuracy = 0.6993 | Macro F1 = 0.6980 | Macro Precision = 0.6993 | Macro Recall = 0.6993 | Sin mejora: 5
Model = LightGBM (metricas_log) N_feat=141       Accuracy = 0.6978 | Macro F1 = 0.6970 | Macro Precision = 0.6987 | Macro Recall = 0.6978 | Sin mejora: 6
Model = LightGBM

## <span style="color:#2ca02c">4.6. Clasificador LightGBM con métricas logs</span>

In [54]:
ALGORITMO_CLASIFICADOR = LGBMClassifier
nombre_algoritmo       ='LightGBM'

### <span style="color:#2ca02c"> 4.6.1. Clasificador LightGBM con métricas logs (3 puntos)</span>

In [48]:
N_puntos = 3
DATOS    = 'metricas_logs 3 puntos'

In [49]:

#hacemos esto para preparar los datos en las mismas variables todos los algoritmos

lista_features = lista_features_names_3puntos

X_train = X_matrix_sel_data(X_matrix,lista_feature_names,lista_idx_filas=idx_train,lista_feature_sel=lista_features)
print(X_train.shape)
X_test  = X_matrix_sel_data(X_matrix,lista_feature_names,lista_idx_filas=idx_test ,lista_feature_sel=lista_features)
print(X_test.shape)
X_val   = X_matrix_sel_data(X_matrix,lista_feature_names,lista_idx_filas=idx_val ,lista_feature_sel=lista_features)
print(X_val.shape)


(10282, 993)
(3445, 993)
(3445, 993)


In [50]:
params_grid = {
    'n_estimators': [200, 300, 400],
    'learning_rate': [0.01, 0.05, 0.1],
    'max_depth': [-1, 6, 8, 10],
    'num_leaves': [31, 50, 63],
    'subsample': [0.7, 0.8, 0.9, 1.0],
    'colsample_bytree': [0.7, 0.8, 0.9, 1.0],
    'min_child_samples': [20, 30, 50],
    'reg_alpha': [0.0, 0.1, 0.5],
    'reg_lambda': [0.0, 0.1, 0.5],
    'verbosity': [-1]
    
}

In [51]:
warnings.filterwarnings('ignore', category=UserWarning)

In [52]:
optuna.logging.set_verbosity(optuna.logging.WARNING)
N_trials = 30

# Barra de progreso global
pbar = tqdm(total=N_trials, desc=f"Optimizando {nombre_algoritmo}")

# Crear el estudio

study = optuna.create_study(
    direction='maximize',
    sampler=TPESampler(seed=SEED) 
)

# Enlazamos DIRECTAMENTE 'objective_universal' en el partial (evita usar alias antiguos)
objective_func = partial(
    objective_universal, 
    config=params_grid,
    clasificador_cls=ALGORITMO_CLASIFICADOR,
    X_train=X_train,
    X_test=X_test,
    Y_train_encoded=Y_train_encoded,
    Y_test_encoded=Y_test_encoded
)

# Ejecutar la optimización
print(f"=== INICIANDO OPTIMIZACIÓN DE {nombre_algoritmo} CON OPTUNA ===")
try:
    study.optimize(objective_func, n_trials=N_trials, callbacks=[print_callback], n_jobs=1)
finally:
    pbar.close()

# Resultados finales
print(f"\n=== MEJOR COMBINACIÓN ENCONTRADA ({nombre_algoritmo}) ===")
print(study.best_params)
print(f"Mejor Accuracy: {study.best_value:.4f}")

# Reconstruir el modelo final con los mejores parámetros óptimos
mejores_params = study.best_params.copy()
mejores_params['random_state'] = SEED
if 'n_jobs' in ALGORITMO_CLASIFICADOR().get_params():
    mejores_params['n_jobs'] = -1

Optimizando LightGBM:   0%|          | 0/30 [00:00<?, ?it/s]

=== INICIANDO OPTIMIZACIÓN DE LightGBM CON OPTUNA ===


Optimizando LightGBM:   3%|▎         | 1/30 [00:23<11:27, 23.70s/it, Mejor_Acc=0.6679, Trial=0]

Trial  0                | Actual Acc: 0.6679 | Parámetros: {'n_estimators': 200, 'learning_rate': 0.01, 'max_depth': 6, 'num_leaves': 31, 'subsample': 0.9, 'colsample_bytree': 0.8, 'min_child_samples': 20, 'reg_alpha': 0.1, 'reg_lambda': 0.1, 'verbosity': -1}
   -> MEJOR HASTA AHORA | Mejor Acc:  0.6679 | Parámetros: {'n_estimators': 200, 'learning_rate': 0.01, 'max_depth': 6, 'num_leaves': 31, 'subsample': 0.9, 'colsample_bytree': 0.8, 'min_child_samples': 20, 'reg_alpha': 0.1, 'reg_lambda': 0.1, 'verbosity': -1}
----------------------------------------------------------------------------------------------------


Optimizando LightGBM:   7%|▋         | 2/30 [01:08<16:50, 36.10s/it, Mejor_Acc=0.6792, Trial=1]

...

Trial 27                | Actual Acc: 0.6940 | Parámetros: {'n_estimators': 400, 'learning_rate': 0.05, 'max_depth': 6, 'num_leaves': 50, 'subsample': 0.8, 'colsample_bytree': 1.0, 'min_child_samples': 50, 'reg_alpha': 0.0, 'reg_lambda': 0.1, 'verbosity': -1}
   -> MEJOR HASTA AHORA | Mejor Acc:  0.6940 | Parámetros: {'n_estimators': 400, 'learning_rate': 0.05, 'max_depth': 6, 'num_leaves': 50, 'subsample': 0.8, 'colsample_bytree': 1.0, 'min_child_samples': 50, 'reg_alpha': 0.0, 'reg_lambda': 0.1, 'verbosity': -1}
----------------------------------------------------------------------------------------------------


Optimizando LightGBM:  97%|█████████▋| 29/30 [23:47<01:14, 74.28s/it, Mejor_Acc=0.6940, Trial=28]

Trial 28                | Actual Acc: 0.6769 | Parámetros: {'n_estimators': 400, 'learning_rate': 0.05, 'max_depth': 6, 'num_leaves': 50, 'subsample': 0.8, 'colsample_bytree': 1.0, 'min_child_samples': 30, 'reg_alpha': 0.1, 'reg_lambda': 0.0, 'verbosity': -1}
   -> MEJOR HASTA AHORA | Mejor Acc:  0.6940 | Parámetros: {'n_estimators': 400, 'learning_rate': 0.05, 'max_depth': 6, 'num_leaves': 50, 'subsample': 0.8, 'colsample_bytree': 1.0, 'min_child_samples': 50, 'reg_alpha': 0.0, 'reg_lambda': 0.1, 'verbosity': -1}
----------------------------------------------------------------------------------------------------


Optimizando LightGBM: 100%|██████████| 30/30 [24:39<00:00, 49.32s/it, Mejor_Acc=0.6940, Trial=29]

Trial 29                | Actual Acc: 0.6708 | Parámetros: {'n_estimators': 400, 'learning_rate': 0.05, 'max_depth': 6, 'num_leaves': 50, 'subsample': 0.8, 'colsample_bytree': 1.0, 'min_child_samples': 20, 'reg_alpha': 0.5, 'reg_lambda': 0.1, 'verbosity': -1}
   -> MEJOR HASTA AHORA | Mejor Acc:  0.6940 | Parámetros: {'n_estimators': 400, 'learning_rate': 0.05, 'max_depth': 6, 'num_leaves': 50, 'subsample': 0.8, 'colsample_bytree': 1.0, 'min_child_samples': 50, 'reg_alpha': 0.0, 'reg_lambda': 0.1, 'verbosity': -1}
----------------------------------------------------------------------------------------------------

=== MEJOR COMBINACIÓN ENCONTRADA (LightGBM) ===
{'n_estimators': 400, 'learning_rate': 0.05, 'max_depth': 6, 'num_leaves': 50, 'subsample': 0.8, 'colsample_bytree': 1.0, 'min_child_samples': 50, 'reg_alpha': 0.0, 'reg_lambda': 0.1, 'verbosity': -1}
Mejor Accuracy: 0.6940


In [53]:
clf_anomalias_opt = ALGORITMO_CLASIFICADOR(**mejores_params)
clf_anomalias_opt.fit(X_train, Y_train_encoded)

,num_leaves,50
,max_depth,6
,learning_rate,0.05
,n_estimators,400
,min_child_samples,50
,subsample,0.8
,reg_lambda,0.1
,random_state,100
,n_jobs,-1
,verbosity,-1
,boosting_type,'gbdt'


In [54]:
model_name=f'{nombre_algoritmo} ({DATOS})(todas las features)'
paquete_modelo={}

paquete_modelo['model_name']         = model_name
paquete_modelo['model']              = clf_anomalias_opt
paquete_modelo['parametros']         = mejores_params
paquete_modelo['lista_features']     = lista_features
paquete_modelo['N_features']         = len(paquete_modelo['lista_features'] )
paquete_modelo['N_features_unicas']  = len(get_lista_feature_sin_lag(paquete_modelo['lista_features'] ))
paquete_modelo['datos']              = DATOS
paquete_modelo['algoritmo']          = nombre_algoritmo

resultado,reporte_dict,Y_pred_probabilidades = evaluar_modelo(paquete_modelo,lista_features,X_test,Y_test_etiquetas,encoder_etiquetas)
print(f"=== REPORTE DE CLASIFICACIÓN DE TIPOS DE ANOMALÍA ===\n{model_name}")
print(paquete_modelo['classification_report_str'])

joblib.dump(paquete_modelo, f'{FOLDER_SALIDA}/models/{model_name}.joblib')

=== REPORTE DE CLASIFICACIÓN DE TIPOS DE ANOMALÍA ===
LightGBM (metricas_logs 3 puntos)(todas las features)
                                               precision    recall  f1-score   support

           access_token_auth_header_error_401       1.00      0.98      0.99        65
          access_token_authorization_form_401       0.46      0.48      0.47        65
         access_token_client_id_not_found_404       1.00      1.00      1.00        65
         access_token_client_secret_wrong_401       1.00      0.98      0.99        65
             access_token_form_urlencoded_400       0.59      0.63      0.61        65
          access_token_illegal_grant_type_400       0.98      1.00      0.99        65
access_token_missing_authorization_header_400       0.54      0.55      0.55        65
     authorization_code_client_id_missing_400       0.90      0.82      0.85        65
     authorization_code_invalid_client_id_404       0.79      0.86      0.82        65
      authorization_c

['result/models/LightGBM (metricas_logs 3 puntos)(todas las features).joblib']

In [55]:
df_importancias_agrupada, importancias, num_cero = obtener_importancias_agrupadas(paquete_modelo['model'].feature_importances_, paquete_modelo['lista_features'])
print(f"Características con importancia cero: {num_cero}")

Características con importancia cero: 56


In [58]:
n_min = 135
n_max = len(importancias) - num_cero


# --- VARIABLES PARA EL EARLY STOPPING ---
paciencia = 15
iteraciones_sin_mejora = 0
mejor_score_global = -1.0
mejor_paquete_modelo = None

for N_feat in range(n_min, n_max + 1):
    model_name = f'{nombre_algoritmo} ({DATOS})   N_feat={N_feat}'
    lista_features_sel = []
    for feature in df_importancias_agrupada['caracteristica'].to_list()[:N_feat]:
        if N_puntos>1:
            for lag in range(0, N_puntos):
                lista_features_sel.append(f'{feature}(lag_{lag})')
        else:
            lista_features_sel.append(feature)

    X_train_sub = X_matrix_sel_data(X_matrix, lista_feature_names, lista_idx_filas=idx_train, lista_feature_sel=lista_features_sel)
    X_test_sub  = X_matrix_sel_data(X_matrix, lista_feature_names, lista_idx_filas=idx_test,  lista_feature_sel=lista_features_sel)
   
    clf_anomalias_opt = ALGORITMO_CLASIFICADOR(**mejores_params)
    clf_anomalias_opt.fit(X_train_sub, Y_train_encoded)

    paquete_modelo = {}
    paquete_modelo['model_name']          = model_name
    paquete_modelo['model']               = clf_anomalias_opt
    paquete_modelo['parametros']          = mejores_params
    paquete_modelo['lista_features']      = lista_features_sel
    paquete_modelo['N_features']          = len(paquete_modelo['lista_features'])
    paquete_modelo['N_features_unicas']   = len(get_lista_feature_sin_lag(paquete_modelo['lista_features']))
    paquete_modelo['datos']               = 'metricas_log_lags'
    paquete_modelo['algoritmo']           = nombre_algoritmo
    
    resultado, reporte_dict, Y_pred_probabilidades = evaluar_modelo(paquete_modelo, lista_features, X_test, Y_test_etiquetas, encoder_etiquetas)
    accuracy        = resultado['accuracy']
    macro_f1        = resultado['f1']
    macro_precision = resultado['precision']
    macro_recall    = resultado['recall']
    


    # --- LÓGICA DE EARLY STOPPING ---
    score_actual = macro_f1 
    
    if score_actual > mejor_score_global:
        mejor_score_global = score_actual
        mejor_paquete_modelo = paquete_modelo.copy()
        iteraciones_sin_mejora = 0  # Reiniciamos contador si mejora
    else:
        iteraciones_sin_mejora += 1

    # --- IMPRESIÓN EN UNA SOLA LÍNEA ---
    print(f"Model = {model_name:40s} Accuracy = {accuracy:.4f} | Macro F1 = {macro_f1:.4f} | Macro Precision = {macro_precision:.4f} | Macro Recall = {macro_recall:.4f} | Mejor Macro F1 = {mejor_score_global:.4f} | Iteraciones_sin_mejora = {iteraciones_sin_mejora}")
    del X_train_sub, X_test_sub
    gc.collect()
    
    # Comprobación de parada
    if iteraciones_sin_mejora >= paciencia:
        print(f"\n[!] Early Stopping activado: No ha habido mejoras en {paciencia} iteraciones consecutivas.")
        print(f"Deteniendo búsqueda. El mejor modelo se logró con {mejor_paquete_modelo['N_features_unicas']} características base.")
        break


Model = LightGBM (metricas_logs 3 puntos)   N_feat=135 Accuracy = 0.6807 | Macro F1 = 0.6779 | Macro Precision = 0.6786 | Macro Recall = 0.6807 | Mejor Macro F1 = 0.6779 | Iteraciones_sin_mejora = 0
Model = LightGBM (metricas_logs 3 puntos)   N_feat=136 Accuracy = 0.6827 | Macro F1 = 0.6812 | Macro Precision = 0.6837 | Macro Recall = 0.6827 | Mejor Macro F1 = 0.6812 | Iteraciones_sin_mejora = 0
Model = LightGBM (metricas_logs 3 puntos)   N_feat=137 Accuracy = 0.6790 | Macro F1 = 0.6779 | Macro Precision = 0.6798 | Macro Recall = 0.6790 | Mejor Macro F1 = 0.6812 | Iteraciones_sin_mejora = 1
Model = LightGBM (metricas_logs 3 puntos)   N_feat=138 Accuracy = 0.6821 | Macro F1 = 0.6810 | Macro Precision = 0.6832 | Macro Recall = 0.6821 | Mejor Macro F1 = 0.6812 | Iteraciones_sin_mejora = 2
Model = LightGBM (metricas_logs 3 puntos)   N_feat=139 Accuracy = 0.6784 | Macro F1 = 0.6777 | Macro Precision = 0.6806 | Macro Recall = 0.6784 | Mejor Macro F1 = 0.6812 | Iteraciones_sin_mejora = 3
Model

In [60]:
paquete_modelo = mejor_paquete_modelo.copy()

mejor_macro_f1_global    = paquete_modelo['resultado']['f1']
mejor_features_global    = paquete_modelo['lista_features'].copy()
pasos_sin_mejora         = 0
limite_puntos_sin_mejora = paciencia + 5

print(f"Iniciando eliminación recursiva desde {paquete_modelo['N_features_unicas']} características.\n")

while paquete_modelo['N_features_unicas'] > 1 and pasos_sin_mejora < limite_puntos_sin_mejora:
    N_feat = paquete_modelo['N_features_unicas']
    
    # Extraer importancias y características actuales
    importancias_raw = paquete_modelo['model'].feature_importances_
    features_raw = paquete_modelo['lista_features']
    
    df_importancias_agrupada, _, num_cero = obtener_importancias_agrupadas(importancias_raw, features_raw)

    # Seleccionar características para el siguiente paso
    lista_features_sel = []
    for feature in df_importancias_agrupada['caracteristica'].to_list()[:(N_feat-1)]:
        if N_puntos>1:
            for lag in range(0, N_puntos):
                lista_features_sel.append(f'{feature}(lag_{lag})')
        else:
            lista_features_sel.append(f'{feature}')
    
    X_train_sub = X_matrix_sel_data(X_matrix, lista_feature_names, lista_idx_filas=idx_train, lista_feature_sel=lista_features_sel)
    X_test_sub  = X_matrix_sel_data(X_matrix, lista_feature_names, lista_idx_filas=idx_test,  lista_feature_sel=lista_features_sel)
   
    clf_anomalias_opt = ALGORITMO_CLASIFICADOR(**mejores_params)
    clf_anomalias_opt.fit(X_train_sub, Y_train_encoded)

    model_name = f'{nombre_algoritmo} (metricas_log) N_feat={N_feat-1}'
    paquete_modelo = {}
    paquete_modelo['model_name']          = model_name
    paquete_modelo['model']               = clf_anomalias_opt
    paquete_modelo['parametros']          = mejores_params
    paquete_modelo['lista_features']      = lista_features_sel
    paquete_modelo['N_features']          = len(paquete_modelo['lista_features'])
    paquete_modelo['N_features_unicas']   = len(get_lista_feature_sin_lag(paquete_modelo['lista_features']))
    paquete_modelo['datos']               = 'metricas_log_lags'
    paquete_modelo['algoritmo']           = nombre_algoritmo
    
    resultado, reporte_dict, Y_pred_probabilidades = evaluar_modelo(paquete_modelo, lista_features, X_test, Y_test_etiquetas, encoder_etiquetas)
   
    accuracy        = resultado['accuracy']
    macro_f1        = resultado['f1']
    macro_precision = resultado['precision']
    macro_recall    = resultado['recall']
    

    
    if macro_f1 >= mejor_macro_f1_global:
        mejor_macro_f1_global = macro_f1
        mejor_features_global = paquete_modelo['lista_features'].copy()
        pasos_sin_mejora = 0  
        mejor_paquete_modelo = paquete_modelo.copy()
    else:
        pasos_sin_mejora += 1  
        
    print(f"Model = {model_name:40s} Accuracy = {accuracy:.4f} | Macro F1 = {macro_f1:.4f} | Macro Precision = {macro_precision:.4f} | Macro Recall = {macro_recall:.4f} | Sin mejora: {pasos_sin_mejora}")
    del X_train_sub, X_test_sub, importancias_raw, df_importancias_agrupada
    gc.collect()

print(f"\n¡Proceso finalizado!")
print(f"Mejor Macro F1 alcanzado: {mejor_macro_f1_global:.4f}")
print(f"Número óptimo de características base: {len(mejor_features_global)}")


Iniciando eliminación recursiva desde 147 características.

Model = LightGBM (metricas_log) N_feat=146       Accuracy = 0.6958 | Macro F1 = 0.6950 | Macro Precision = 0.6972 | Macro Recall = 0.6958 | Sin mejora: 1
Model = LightGBM (metricas_log) N_feat=145       Accuracy = 0.6958 | Macro F1 = 0.6934 | Macro Precision = 0.6940 | Macro Recall = 0.6958 | Sin mejora: 2
Model = LightGBM (metricas_log) N_feat=144       Accuracy = 0.6926 | Macro F1 = 0.6919 | Macro Precision = 0.6943 | Macro Recall = 0.6926 | Sin mejora: 3
Model = LightGBM (metricas_log) N_feat=143       Accuracy = 0.6958 | Macro F1 = 0.6944 | Macro Precision = 0.6964 | Macro Recall = 0.6958 | Sin mejora: 4
Model = LightGBM (metricas_log) N_feat=142       Accuracy = 0.6993 | Macro F1 = 0.6980 | Macro Precision = 0.6993 | Macro Recall = 0.6993 | Sin mejora: 5
Model = LightGBM (metricas_log) N_feat=141       Accuracy = 0.6978 | Macro F1 = 0.6970 | Macro Precision = 0.6987 | Macro Recall = 0.6978 | Sin mejora: 6
Model = LightGBM

In [61]:
mejor_paquete_modelo['model_name']

'LightGBM (metricas_logs 3 puntos)   N_feat=147'

In [62]:
joblib.dump(mejor_paquete_modelo, f"{FOLDER_SALIDA}/models/{mejor_paquete_modelo['model_name']}.joblib")

['result/models/LightGBM (metricas_logs 3 puntos)   N_feat=147.joblib']

### <span style="color:#2ca02c">4.5.2. Clasificador LightGBM (1 punto de 3 intervalos) con metricas_logs</span>

In [55]:
N_puntos = 1
DATOS    = 'metricas_logs 1 punto suma'

In [56]:
lista_features = lista_feature_names_1punto_sum

X_train = X_matrix_sel_data(X_matrix,lista_feature_names,lista_idx_filas=idx_train,lista_feature_sel=lista_features)
print(X_train.shape)
X_test  = X_matrix_sel_data(X_matrix,lista_feature_names,lista_idx_filas=idx_test ,lista_feature_sel=lista_features)
print(X_test.shape)
X_val   = X_matrix_sel_data(X_matrix,lista_feature_names,lista_idx_filas=idx_val ,lista_feature_sel=lista_features)
print(X_val.shape)

(10282, 331)
(3445, 331)
(3445, 331)


In [57]:
params_grid = {
    'n_estimators': [200, 300, 400],
    'learning_rate': [0.01, 0.05, 0.1],
    'max_depth': [-1, 6, 8, 10],
    'num_leaves': [31, 50, 63],
    'subsample': [0.7, 0.8, 0.9, 1.0],
    'colsample_bytree': [0.7, 0.8, 0.9, 1.0],
    'min_child_samples': [20, 30, 50],
    'reg_alpha': [0.0, 0.1, 0.5],
    'reg_lambda': [0.0, 0.1, 0.5],
    'verbosity': [-1]
    
}

In [53]:

optuna.logging.set_verbosity(optuna.logging.WARNING)
N_trials = 30

# Barra de progreso global
pbar = tqdm(total=N_trials, desc=f"Optimizando {nombre_algoritmo}")

# Crear el estudio

study = optuna.create_study(
    direction='maximize',
    sampler=TPESampler(seed=SEED) 
)

# Enlazamos DIRECTAMENTE 'objective_universal' en el partial (evita usar alias antiguos)
objective_func = partial(
    objective_universal, 
    config=params_grid,
    clasificador_cls=ALGORITMO_CLASIFICADOR,
    X_train=X_train,
    X_test=X_test,
    Y_train_encoded=Y_train_encoded,
    Y_test_encoded=Y_test_encoded
)

# Ejecutar la optimización
print(f"=== INICIANDO OPTIMIZACIÓN DE {nombre_algoritmo} CON OPTUNA ===")
try:
    study.optimize(objective_func, n_trials=N_trials, callbacks=[print_callback], n_jobs=1)
finally:
    pbar.close()

# Resultados finales
print(f"\n=== MEJOR COMBINACIÓN ENCONTRADA ({nombre_algoritmo}) ===")
print(study.best_params)
print(f"Mejor Accuracy: {study.best_value:.4f}")

# Reconstruir el modelo final con los mejores parámetros óptimos
mejores_params = study.best_params.copy()
mejores_params['random_state'] = SEED
if 'n_jobs' in ALGORITMO_CLASIFICADOR().get_params():
    mejores_params['n_jobs'] = -1
	
	

Optimizando LightGBM:   0%|          | 0/30 [00:00<?, ?it/s]

=== INICIANDO OPTIMIZACIÓN DE LightGBM CON OPTUNA ===


Optimizando LightGBM:   3%|▎         | 1/30 [00:17<08:39, 17.92s/it, Mejor_Acc=0.6424, Trial=0]

Trial  0                | Actual Acc: 0.6424 | Parámetros: {'n_estimators': 200, 'learning_rate': 0.01, 'max_depth': 6, 'num_leaves': 31, 'subsample': 0.9, 'colsample_bytree': 0.8, 'min_child_samples': 20, 'reg_alpha': 0.1, 'reg_lambda': 0.1, 'verbosity': -1}
   -> MEJOR HASTA AHORA | Mejor Acc:  0.6424 | Parámetros: {'n_estimators': 200, 'learning_rate': 0.01, 'max_depth': 6, 'num_leaves': 31, 'subsample': 0.9, 'colsample_bytree': 0.8, 'min_child_samples': 20, 'reg_alpha': 0.1, 'reg_lambda': 0.1, 'verbosity': -1}
----------------------------------------------------------------------------------------------------


Optimizando LightGBM:   7%|▋         | 2/30 [00:51<12:43, 27.28s/it, Mejor_Acc=0.6496, Trial=1]

...

Trial 27                | Actual Acc: 0.6517 | Parámetros: {'n_estimators': 400, 'learning_rate': 0.05, 'max_depth': 6, 'num_leaves': 50, 'subsample': 0.7, 'colsample_bytree': 1.0, 'min_child_samples': 50, 'reg_alpha': 0.0, 'reg_lambda': 0.0, 'verbosity': -1}
   -> MEJOR HASTA AHORA | Mejor Acc:  0.6621 | Parámetros: {'n_estimators': 400, 'learning_rate': 0.05, 'max_depth': 8, 'num_leaves': 50, 'subsample': 0.7, 'colsample_bytree': 1.0, 'min_child_samples': 50, 'reg_alpha': 0.0, 'reg_lambda': 0.0, 'verbosity': -1}
----------------------------------------------------------------------------------------------------


Optimizando LightGBM:  97%|█████████▋| 29/30 [13:53<00:32, 32.57s/it, Mejor_Acc=0.6621, Trial=28]

Trial 28                | Actual Acc: 0.6491 | Parámetros: {'n_estimators': 400, 'learning_rate': 0.05, 'max_depth': 10, 'num_leaves': 50, 'subsample': 0.7, 'colsample_bytree': 1.0, 'min_child_samples': 30, 'reg_alpha': 0.1, 'reg_lambda': 0.0, 'verbosity': -1}
   -> MEJOR HASTA AHORA | Mejor Acc:  0.6621 | Parámetros: {'n_estimators': 400, 'learning_rate': 0.05, 'max_depth': 8, 'num_leaves': 50, 'subsample': 0.7, 'colsample_bytree': 1.0, 'min_child_samples': 50, 'reg_alpha': 0.0, 'reg_lambda': 0.0, 'verbosity': -1}
----------------------------------------------------------------------------------------------------


Optimizando LightGBM: 100%|██████████| 30/30 [14:10<00:00, 28.36s/it, Mejor_Acc=0.6621, Trial=29]

Trial 29                | Actual Acc: 0.6438 | Parámetros: {'n_estimators': 400, 'learning_rate': 0.05, 'max_depth': 6, 'num_leaves': 50, 'subsample': 1.0, 'colsample_bytree': 0.8, 'min_child_samples': 20, 'reg_alpha': 0.5, 'reg_lambda': 0.0, 'verbosity': -1}
   -> MEJOR HASTA AHORA | Mejor Acc:  0.6621 | Parámetros: {'n_estimators': 400, 'learning_rate': 0.05, 'max_depth': 8, 'num_leaves': 50, 'subsample': 0.7, 'colsample_bytree': 1.0, 'min_child_samples': 50, 'reg_alpha': 0.0, 'reg_lambda': 0.0, 'verbosity': -1}
----------------------------------------------------------------------------------------------------

=== MEJOR COMBINACIÓN ENCONTRADA (LightGBM) ===
{'n_estimators': 400, 'learning_rate': 0.05, 'max_depth': 8, 'num_leaves': 50, 'subsample': 0.7, 'colsample_bytree': 1.0, 'min_child_samples': 50, 'reg_alpha': 0.0, 'reg_lambda': 0.0, 'verbosity': -1}
Mejor Accuracy: 0.6621


In [47]:
clf_anomalias_opt = ALGORITMO_CLASIFICADOR(**mejores_params)
clf_anomalias_opt.fit(X_train, Y_train_encoded)

NameError: name 'ALGORITMO_CLASIFICADOR' is not defined

In [68]:
model_name=f'{nombre_algoritmo} ({DATOS})(todas las features)'
paquete_modelo={}

paquete_modelo['model_name']         = model_name
paquete_modelo['model']              = clf_anomalias_opt
paquete_modelo['parametros']         = mejores_params
paquete_modelo['lista_features']     = lista_features
paquete_modelo['N_features']         = len(paquete_modelo['lista_features'] )
paquete_modelo['N_features_unicas']  = len(get_lista_feature_sin_lag(paquete_modelo['lista_features'] ))
paquete_modelo['datos']              = DATOS
paquete_modelo['algoritmo']          = nombre_algoritmo

resultado,reporte_dict,Y_pred_probabilidades = evaluar_modelo(paquete_modelo,lista_features,X_test,Y_test_etiquetas,encoder_etiquetas)
print(f"=== REPORTE DE CLASIFICACIÓN DE TIPOS DE ANOMALÍA ===\n{model_name}")
print(paquete_modelo['classification_report_str'])

joblib.dump(paquete_modelo, f'{FOLDER_SALIDA}/models/{model_name}.joblib')

=== REPORTE DE CLASIFICACIÓN DE TIPOS DE ANOMALÍA ===
LightGBM (metricas_logs 1 punto suma)(todas las features)
                                               precision    recall  f1-score   support

           access_token_auth_header_error_401       0.94      0.98      0.96        65
          access_token_authorization_form_401       0.49      0.62      0.54        65
         access_token_client_id_not_found_404       1.00      0.98      0.99        65
         access_token_client_secret_wrong_401       0.98      0.98      0.98        65
             access_token_form_urlencoded_400       0.70      0.66      0.68        65
          access_token_illegal_grant_type_400       0.98      1.00      0.99        65
access_token_missing_authorization_header_400       0.50      0.46      0.48        65
     authorization_code_client_id_missing_400       0.84      0.83      0.84        65
     authorization_code_invalid_client_id_404       0.79      0.80      0.79        65
      authorizati

['result/models/LightGBM (metricas_logs 1 punto suma)(todas las features).joblib']

In [69]:
df_importancias_agrupada, importancias, num_cero = obtener_importancias_agrupadas(paquete_modelo['model'].feature_importances_, paquete_modelo['lista_features'])
print(f"Características con importancia cero: {num_cero}")

Características con importancia cero: 53


In [72]:
n_min = 172
n_max = len(importancias) - num_cero


# --- VARIABLES PARA EL EARLY STOPPING ---
paciencia = 15
iteraciones_sin_mejora = 0
mejor_score_global = -1.0
mejor_paquete_modelo = None

for N_feat in range(n_min, n_max + 1):
    model_name = f'{nombre_algoritmo} ({DATOS})   N_feat={N_feat}'
    lista_features_sel = []
    for feature in df_importancias_agrupada['caracteristica'].to_list()[:N_feat]:
        if N_puntos>1:
            for lag in range(0, N_puntos):
                lista_features_sel.append(f'{feature}(lag_{lag})')
        else:
            lista_features_sel.append(feature)

    X_train_sub = X_matrix_sel_data(X_matrix, lista_feature_names, lista_idx_filas=idx_train, lista_feature_sel=lista_features_sel)
    X_test_sub  = X_matrix_sel_data(X_matrix, lista_feature_names, lista_idx_filas=idx_test,  lista_feature_sel=lista_features_sel)
   
    clf_anomalias_opt = ALGORITMO_CLASIFICADOR(**mejores_params)
    clf_anomalias_opt.fit(X_train_sub, Y_train_encoded)

    paquete_modelo = {}
    paquete_modelo['model_name']          = model_name
    paquete_modelo['model']               = clf_anomalias_opt
    paquete_modelo['parametros']          = mejores_params
    paquete_modelo['lista_features']      = lista_features_sel
    paquete_modelo['N_features']          = len(paquete_modelo['lista_features'])
    paquete_modelo['N_features_unicas']   = len(get_lista_feature_sin_lag(paquete_modelo['lista_features']))
    paquete_modelo['datos']               = 'metricas_log_lags'
    paquete_modelo['algoritmo']           = nombre_algoritmo
    
    resultado, reporte_dict, Y_pred_probabilidades = evaluar_modelo(paquete_modelo, lista_features, X_test, Y_test_etiquetas, encoder_etiquetas)
    accuracy        = resultado['accuracy']
    macro_f1        = resultado['f1']
    macro_precision = resultado['precision']
    macro_recall    = resultado['recall']
    


    # --- LÓGICA DE EARLY STOPPING ---
    score_actual = macro_f1 
    
    if score_actual > mejor_score_global:
        mejor_score_global = score_actual
        mejor_paquete_modelo = paquete_modelo.copy()
        iteraciones_sin_mejora = 0  # Reiniciamos contador si mejora
    else:
        iteraciones_sin_mejora += 1

    # --- IMPRESIÓN EN UNA SOLA LÍNEA ---
    print(f"Model = {model_name:40s} Accuracy = {accuracy:.4f} | Macro F1 = {macro_f1:.4f} | Macro Precision = {macro_precision:.4f} | Macro Recall = {macro_recall:.4f} | Mejor Macro F1 = {mejor_score_global:.4f} | Iteraciones_sin_mejora = {iteraciones_sin_mejora}")
    del X_train_sub, X_test_sub
    gc.collect()
    
    # Comprobación de parada
    if iteraciones_sin_mejora >= paciencia:
        print(f"\n[!] Early Stopping activado: No ha habido mejoras en {paciencia} iteraciones consecutivas.")
        print(f"Deteniendo búsqueda. El mejor modelo se logró con {mejor_paquete_modelo['N_features_unicas']} características base.")
        break


Model = LightGBM (metricas_logs 1 punto suma)   N_feat=172 Accuracy = 0.6543 | Macro F1 = 0.6525 | Macro Precision = 0.6575 | Macro Recall = 0.6543 | Mejor Macro F1 = 0.6525 | Iteraciones_sin_mejora = 0
Model = LightGBM (metricas_logs 1 punto suma)   N_feat=173 Accuracy = 0.6528 | Macro F1 = 0.6505 | Macro Precision = 0.6545 | Macro Recall = 0.6528 | Mejor Macro F1 = 0.6525 | Iteraciones_sin_mejora = 1
Model = LightGBM (metricas_logs 1 punto suma)   N_feat=174 Accuracy = 0.6537 | Macro F1 = 0.6512 | Macro Precision = 0.6547 | Macro Recall = 0.6537 | Mejor Macro F1 = 0.6525 | Iteraciones_sin_mejora = 2
Model = LightGBM (metricas_logs 1 punto suma)   N_feat=175 Accuracy = 0.6499 | Macro F1 = 0.6481 | Macro Precision = 0.6519 | Macro Recall = 0.6499 | Mejor Macro F1 = 0.6525 | Iteraciones_sin_mejora = 3
Model = LightGBM (metricas_logs 1 punto suma)   N_feat=176 Accuracy = 0.6522 | Macro F1 = 0.6498 | Macro Precision = 0.6526 | Macro Recall = 0.6522 | Mejor Macro F1 = 0.6525 | Iteraciones_

In [73]:
mejor_paquete_modelo['model_name']

'LightGBM (metricas_logs 1 punto suma)   N_feat=202'

In [74]:
joblib.dump(mejor_paquete_modelo, f"{FOLDER_SALIDA}/models/{mejor_paquete_modelo['model_name']}.joblib")

['result/models/LightGBM (metricas_logs 1 punto suma)   N_feat=202.joblib']

In [81]:
paquete_modelo = mejor_paquete_modelo.copy()

mejor_macro_f1_global    = paquete_modelo['resultado']['f1']
mejor_features_global    = paquete_modelo['lista_features'].copy()
pasos_sin_mejora         = 0
limite_puntos_sin_mejora = paciencia + 5

print(f"Iniciando eliminación recursiva desde {paquete_modelo['N_features_unicas']} características.\n")

while paquete_modelo['N_features_unicas'] > 1 and pasos_sin_mejora < limite_puntos_sin_mejora:
    N_feat = paquete_modelo['N_features_unicas']
    
    # Extraer importancias y características actuales
    importancias_raw = paquete_modelo['model'].feature_importances_
    features_raw = paquete_modelo['lista_features']
    
    df_importancias_agrupada, _, num_cero = obtener_importancias_agrupadas(importancias_raw, features_raw)

    # Seleccionar características para el siguiente paso
    lista_features_sel = []
    for feature in df_importancias_agrupada['caracteristica'].to_list()[:(N_feat-1)]:
        if N_puntos>1:
            for lag in range(0, N_puntos):
                lista_features_sel.append(f'{feature}(lag_{lag})')
        else:
            lista_features_sel.append(f'{feature}')

        
    X_train_sub = X_matrix_sel_data(X_matrix, lista_feature_names, lista_idx_filas=idx_train, lista_feature_sel=lista_features_sel)
    X_test_sub  = X_matrix_sel_data(X_matrix, lista_feature_names, lista_idx_filas=idx_test,  lista_feature_sel=lista_features_sel)
   
    clf_anomalias_opt = ALGORITMO_CLASIFICADOR(**mejores_params)
    clf_anomalias_opt.fit(X_train_sub, Y_train_encoded)

    model_name = f'{nombre_algoritmo} (metricas_log) N_feat={N_feat-1}'
    paquete_modelo = {}
    paquete_modelo['model_name']          = model_name
    paquete_modelo['model']               = clf_anomalias_opt
    paquete_modelo['parametros']          = mejores_params
    paquete_modelo['lista_features']      = lista_features_sel
    paquete_modelo['N_features']          = len(paquete_modelo['lista_features'])
    paquete_modelo['N_features_unicas']   = len(get_lista_feature_sin_lag(paquete_modelo['lista_features']))
    paquete_modelo['datos']               = 'metricas_log_lags'
    paquete_modelo['algoritmo']           = nombre_algoritmo
    
    resultado, reporte_dict, Y_pred_probabilidades = evaluar_modelo(paquete_modelo, lista_features, X_test, Y_test_etiquetas, encoder_etiquetas)
   
    accuracy        = resultado['accuracy']
    macro_f1        = resultado['f1']
    macro_precision = resultado['precision']
    macro_recall    = resultado['recall']
    

    
    if macro_f1 >= mejor_macro_f1_global:
        mejor_macro_f1_global = macro_f1
        mejor_features_global = paquete_modelo['lista_features'].copy()
        pasos_sin_mejora = 0  
        mejor_paquete_modelo = paquete_modelo.copy()
    else:
        pasos_sin_mejora += 1  
        
    print(f"Model = {model_name:40s} Accuracy = {accuracy:.4f} | Macro F1 = {macro_f1:.4f} | Macro Precision = {macro_precision:.4f} | Macro Recall = {macro_recall:.4f} | Sin mejora: {pasos_sin_mejora}")
    del X_train_sub, X_test_sub, importancias_raw, df_importancias_agrupada
    gc.collect()

print(f"\n¡Proceso finalizado!")
print(f"Mejor Macro F1 alcanzado: {mejor_macro_f1_global:.4f}")
print(f"Número óptimo de características base: {len(mejor_features_global)}")


Iniciando eliminación recursiva desde 202 características.

Model = LightGBM (metricas_log) N_feat=201       Accuracy = 0.6612 | Macro F1 = 0.6584 | Macro Precision = 0.6617 | Macro Recall = 0.6612 | Sin mejora: 1
Model = LightGBM (metricas_log) N_feat=200       Accuracy = 0.6621 | Macro F1 = 0.6597 | Macro Precision = 0.6621 | Macro Recall = 0.6621 | Sin mejora: 2
Model = LightGBM (metricas_log) N_feat=199       Accuracy = 0.6650 | Macro F1 = 0.6623 | Macro Precision = 0.6646 | Macro Recall = 0.6650 | Sin mejora: 3
Model = LightGBM (metricas_log) N_feat=198       Accuracy = 0.6615 | Macro F1 = 0.6590 | Macro Precision = 0.6616 | Macro Recall = 0.6615 | Sin mejora: 4
Model = LightGBM (metricas_log) N_feat=197       Accuracy = 0.6610 | Macro F1 = 0.6589 | Macro Precision = 0.6629 | Macro Recall = 0.6610 | Sin mejora: 5
Model = LightGBM (metricas_log) N_feat=196       Accuracy = 0.6578 | Macro F1 = 0.6553 | Macro Precision = 0.6575 | Macro Recall = 0.6578 | Sin mejora: 6
Model = LightGBM

In [62]:
mejor_paquete_modelo = joblib.load('result/models/LightGBM (metricas_logs 1 punto suma) N_feat=192.joblib')
mejores_params=mejor_paquete_modelo['parametros']
paquete_modelo = mejor_paquete_modelo.copy()
print('Resultado del mejor modelo de partida:')
print(mejor_paquete_modelo['resultado'])
print(f'N_features = {mejor_paquete_modelo["N_features"]}')
mejor_macro_f1_global=paquete_modelo['resultado']['f1']


paquete_modelo = mejor_paquete_modelo.copy()
mejor_features_global    = paquete_modelo['lista_features'].copy()

pasos_sin_mejora         = 0
limite_puntos_sin_mejora = 20
ratio_prueba             = 0.1

print(f"Iniciando eliminación recursiva inteligente (10% peor) desde {paquete_modelo['N_features_unicas']} características.\n")

while paquete_modelo['N_features_unicas'] > 1 and pasos_sin_mejora < limite_puntos_sin_mejora:
    N_feat_actual = paquete_modelo['N_features_unicas']
    
    # 1. Extraer importancias y ordenarlas de mayor a menor
    importancias_raw = paquete_modelo['model'].feature_importances_
    features_raw = paquete_modelo['lista_features']
    df_importancias_agrupada, _, num_cero = obtener_importancias_agrupadas(importancias_raw, features_raw)
    
    lista_features_base_ordenada = df_importancias_agrupada['caracteristica'].to_list()
    
    # Definir cuántas candidatas por la cola vamos a probar (el 10%, mínimo 1, máximo las disponibles)
    num_candidatas_quitar = max(1, int(N_feat_actual * ratio_prueba))
    # Nos aseguramos de no intentar quitar más de las que hay
    num_candidatas_quitar = min(num_candidatas_quitar, N_feat_actual - 1)
    
    print(f"\n--- Evaluando reducción desde {N_feat_actual} features. Probando las {num_candidatas_quitar} peores candidatas ---")
    
    mejor_paquete_iteracion = None
    mejor_f1_iteracion = -1.0
    
    # 2. Probar a quitar una a una las peores candidatas (las que están al final de la lista)
    mejor_intento = 0
    for i in range(num_candidatas_quitar):
        intento_num = i + 1
        print(f"  [Intento {intento_num:3d}/{num_candidatas_quitar:3d}] Probando modelo de {N_feat_actual - 1} features...", end=" ")
        
        # Creamos la lista de características base quitando la candidata de la posición desde el final
        # i=0 quita la peores de todas, i=1 quita la penúltima peor, etc.
        indice_a_excluir = len(lista_features_base_ordenada) - 1 - i
        features_base_prueba = [f for idx, f in enumerate(lista_features_base_ordenada) if idx != indice_a_excluir]
        
        # Generar lista de features con sus lags
        lista_features_sel = []
        for feature in features_base_prueba:
            if N_puntos > 1:
                for lag in range(0, N_puntos):
                    lista_features_sel.append(f'{feature}(lag_{lag})')
            else:
                lista_features_sel.append(f'{feature}')
                
        # Entrenar modelo de prueba
        X_train_sub = X_matrix_sel_data(X_matrix, lista_feature_names, lista_idx_filas=idx_train, lista_feature_sel=lista_features_sel)
        X_test_sub  = X_matrix_sel_data(X_matrix, lista_feature_names, lista_idx_filas=idx_test,  lista_feature_sel=lista_features_sel)
       
        clf_anomalias_opt = ALGORITMO_CLASIFICADOR(**mejores_params)
        clf_anomalias_opt.fit(X_train_sub, Y_train_encoded)

        model_name = f'{nombre_algoritmo} ({DATOS}) N_feat={N_feat_actual - 1}'
        paquete_prueba = {}
        paquete_prueba['model_name']          = model_name
        paquete_prueba['model']               = clf_anomalias_opt
        paquete_prueba['parametros']          = mejores_params
        paquete_prueba['lista_features']      = lista_features_sel
        paquete_prueba['N_features']          = len(paquete_prueba['lista_features'])
        paquete_prueba['N_features_unicas']   = len(get_lista_feature_sin_lag(paquete_prueba['lista_features']))
        paquete_prueba['datos']               = 'metricas_log_lags'
        paquete_prueba['algoritmo']           = nombre_algoritmo
        
        resultado, reporte_dict, Y_pred_probabilidades = evaluar_modelo(paquete_prueba, lista_features, X_test, Y_test_etiquetas, encoder_etiquetas)
        
        acc_p   = resultado['accuracy']
        f1_p    = resultado['f1']
        prec_p  = resultado['precision']
        rec_p   = resultado['recall']
        
        paquete_prueba['resultado'] = resultado
        
        print(f"F1 = {f1_p:.4f} (Acc = {acc_p:.4f})")
        
        del X_train_sub, X_test_sub
        gc.collect()
        
        # Guardar si es el mejor de este subgrupo de pruebas
        if f1_p > mejor_f1_iteracion:
            mejor_intento = intento_num
            mejor_f1_iteracion = f1_p
            mejor_paquete_iteracion = paquete_prueba.copy()
            if mejor_f1_iteracion >= mejor_macro_f1_global:
                break
            


    
    print(f'Mejor intento = {mejor_intento}')
    # 3. Comprobar si el mejor de los modelos reducidos mejora o iguala al global actual
    if mejor_f1_iteracion >= mejor_macro_f1_global:
        mejor_macro_f1_global = mejor_f1_iteracion
        mejor_features_global = mejor_paquete_iteracion['lista_features'].copy()
        pasos_sin_mejora = 0  
        paquete_modelo = mejor_paquete_iteracion.copy()
        mejor_paquete_modelo = paquete_modelo.copy()
        print(f"-> ¡Mejora encontrada! Nuevo mejor Macro F1 global: {mejor_macro_f1_global:.4f} con {paquete_modelo['N_features_unicas']} features.")
    else:
        # Si ninguno del 10% mejoró al padre, aumentamos el contador de pasos sin mejora pero avanzamos con el mejor intento o paramos
        pasos_sin_mejora += 1
        # Opcional: penalizamos avanzando al mejor de los probados para no estancarnos en bucle infinito
        paquete_modelo = mejor_paquete_iteracion.copy()
        print(f"-> Sin mejora global en este ciclo. Pasos sin mejora: {pasos_sin_mejora}")
        
    resultado = paquete_modelo['resultado']
    accuracy        = resultado['accuracy']
    macro_f1        = resultado['f1']
    macro_precision = resultado['precision']
    macro_recall    = resultado['recall']
    print(f"Model = {model_name:40s} Accuracy = {accuracy:.4f} | Macro F1 = {macro_f1:.4f} | Macro Precision = {macro_precision:.4f} | Macro Recall = {macro_recall:.4f}")

print(f"\n¡Proceso finalizado!")
print(f"Mejor Macro F1 alcanzado: {mejor_macro_f1_global:.4f}")
print(f"Número óptimo de características base: {len(mejor_features_global)}")

Resultado del mejor modelo de partida:
{'accuracy': 0.6679245283018868, 'precision': 0.6682365199654308, 'recall': 0.6679245283018868, 'f1': 0.6654290065318454}
N_features = 192
Iniciando eliminación recursiva inteligente (10% peor) desde 192 características.


--- Evaluando reducción desde 192 features. Probando las 19 peores candidatas ---
  [Intento   1/ 19] Probando modelo de 191 features... F1 = 0.6585 (Acc = 0.6615)
  [Intento   2/ 19] Probando modelo de 191 features... F1 = 0.6597 (Acc = 0.6624)
  [Intento   3/ 19] Probando modelo de 191 features... F1 = 0.6610 (Acc = 0.6642)
  [Intento   4/ 19] Probando modelo de 191 features... F1 = 0.6626 (Acc = 0.6644)
  [Intento   5/ 19] Probando modelo de 191 features... F1 = 0.6585 (Acc = 0.6610)
  [Intento   6/ 19] Probando modelo de 191 features... F1 = 0.6611 (Acc = 0.6630)
  [Intento   7/ 19] Probando modelo de 191 features... F1 = 0.6575 (Acc = 0.6610)
  [Intento   8/ 19] Probando modelo de 191 features... F1 = 0.6601 (Acc = 0.6633)


In [63]:
mejor_paquete_modelo['model_name']

'LightGBM (metricas_logs 1 punto suma) N_feat=173'

In [64]:
joblib.dump(mejor_paquete_modelo, f"{FOLDER_SALIDA}/models/{mejor_paquete_modelo['model_name']}.joblib")

['result/models/LightGBM (metricas_logs 1 punto suma) N_feat=173.joblib']

### <span style="color:#2ca02c">4.7. Red neuronal</span>

#### <span style="color:#2ca02c">4.7.1. Calculo de la Red neuronal</span>

In [ ]:
scaler = StandardScaler()
X_anomalia_train_std = scaler.fit_transform(X_anomalia_train)
X_anomalia_test_std = scaler.transform(X_anomalia_test)


X_train_real, X_val_real, Y_train_real, Y_val_real = train_test_split(
    X_anomalia_train_std, 
    Y_train_encoded, 
    test_size=0.2, 
    random_state=SEED, 
    shuffle=True,
    stratify=Y_train_encoded # Opcional pero muy recomendado para 53 clases (mantiene la proporción de cada clase)
)

In [ ]:
optuna.logging.set_verbosity(optuna.logging.WARNING)

N_trials = 16

# 1. Crear una barra de progreso global basada en el número total de trials
pbar = tqdm(total=N_trials, desc="Optimizando Red Neuronal")

# 2. Callback de verbose para ver el progreso de Optuna en tiempo real
def print_callback(study, trial):
    print(f"Trial {trial.number} finalizado | Mejor Loss actual: {study.best_value:.4f} | Parámetros: {trial.params}")
    pbar.update(1)
    pbar.set_postfix({
        "Mejor_Loss": f"{study.best_value:.4f}",
        "Trial": trial.number
    })

# 3. Función objetivo de Optuna recibiendo el diccionario externo de configuración
def objective(trial, config):
    num_clases = len(lista_label) - 1
    
    # Sugerir parámetros utilizando el diccionario de configuración
    n_layers = trial.suggest_categorical('n_layers', config['n_layers'])
    lr = trial.suggest_categorical('lr', config['lr'])
    batch_size = trial.suggest_categorical('batch_size', config['batch_size'])
    dropout_rate = trial.suggest_categorical('dropout_rate', config['dropout_rate'])
    
    # Construcción dinámica de la arquitectura
    model = Sequential()
    model.add(Input(shape=(X_anomalia_train.shape[1],)))
    
    for i in range(n_layers):
        units = trial.suggest_categorical(f'units_l{i}', config['units'])
        model.add(Dense(units))
        model.add(BatchNormalization())
        model.add(tf.keras.layers.Activation('relu'))
        model.add(Dropout(dropout_rate))
        
    model.add(Dense(num_clases, activation='softmax'))
    
    optimizer = tf.keras.optimizers.Adam(learning_rate=lr)
    
    model.compile(
        optimizer=optimizer,
        loss='sparse_categorical_crossentropy',
        metrics=['accuracy']
    )
    
    callbacks = [
        EarlyStopping(monitor='val_loss', patience=10, restore_best_weights=True, verbose=0),
        ReduceLROnPlateau(monitor='val_loss', factor=0.5, patience=5, min_lr=1e-5, verbose=0)
    ]
    
    history = model.fit(
        X_train_real,
        Y_train_real,
        validation_data=(X_val_real, Y_val_real),
        epochs=50,
        batch_size=batch_size,
        callbacks=callbacks,
        verbose=0
    )
    
    val_loss = min(history.history['val_loss'])
    trial.set_user_attr('val_loss', val_loss)
    return val_loss

# 4. Definición externa de los hiperparámetros a probar para la Red Neuronal
params_grid_nn = {
    'n_layers': [1, 2, 3],
    'units': [64, 128, 256, 512],
    'lr': [1e-4, 5e-4, 1e-3, 5e-3],
    'batch_size': [32, 64, 128, 256],
    'dropout_rate': [0.1, 0.2, 0.3, 0.5]
}

# 5. Crear el estudio
study_nn = optuna.create_study(direction='minimize')

# 6. Enlazar la función con los parámetros mediante partial
objective_func_nn = partial(objective, config=params_grid_nn)

# 7. Ejecutar pasando el callback de verbose
print("=== INICIANDO OPTIMIZACIÓN DE RED NEURONAL CON OPTUNA ===")
try:
    study_nn.optimize(objective_func_nn, n_trials=N_trials, callbacks=[print_callback], n_jobs=1)
finally:
    pbar.close()

# 8. Resultados finales y entrenamiento del modelo definitivo
print("\n=== MEJOR COMBINACIÓN ENCONTRADA (RED NEURONAL) ===")
print(study_nn.best_params)
print(f"Mejor Val Loss: {study_nn.best_value:.4f}")


In [ ]:

# Reconstruir el modelo final con los mejores parámetros óptimos
best_params_nn = study_nn.best_params
num_clases = len(lista_label) - 1

best_model = Sequential()
best_model.add(Input(shape=(X_anomalia_train.shape[1],)))

for i in range(best_params_nn['n_layers']):
    units = best_params_nn[f'units_l{i}']
    best_model.add(Dense(units))
    best_model.add(BatchNormalization())
    best_model.add(tf.keras.layers.Activation('relu'))
    best_model.add(Dropout(best_params_nn['dropout_rate']))

best_model.add(Dense(num_clases, activation='softmax'))

best_optimizer = tf.keras.optimizers.Adam(learning_rate=best_params_nn['lr'])

best_model.compile(
    optimizer=best_optimizer,
    loss='sparse_categorical_crossentropy',
    metrics=['accuracy']
)

callbacks_final = [
    EarlyStopping(monitor='val_loss', patience=15, restore_best_weights=True, verbose=1),
    ReduceLROnPlateau(monitor='val_loss', factor=0.5, patience=5, min_lr=1e-5, verbose=1)
]

history_final = best_model.fit(
    X_train_real,
    Y_train_real,
    validation_data=(X_val_real, Y_val_real),
    epochs=300,
    batch_size=best_params_nn['batch_size'],
    callbacks=callbacks_final,
    verbose=1
)

In [ ]:
# 1. Generar predicciones de probabilidad y convertir a índices de clase
Y_pred_prob = best_model.predict(X_anomalia_test_std)  # O X_anomalia_test según el nombre de tu conjunto de test
Y_pred_encoded = np.argmax(Y_pred_prob, axis=1)

# 2. Revertir el codificador de etiquetas para obtener los nombres originales (texto)
Y_pred_tipos = encoder_etiquetas.inverse_transform(Y_pred_encoded)

# 3. Evaluar el rendimiento del clasificador usando las etiquetas reales de test
print("=== REPORTE DE CLASIFICACIÓN DE TIPOS DE ANOMALÍA (Red Neuronal) ===")
print(classification_report(Y_test_etiquetas, Y_pred_tipos))

In [ ]:


model_name='Clasificador_Red_Neuronal'
paquete_NN = {
    'model': clf_anomalias_opt,                
    'metrics_list': lista_metric_name_vector,  # La lista con los nombres de las características
    'model_name': model_name, # Un identificador o nombre descriptivo
    'encoder_etiquetas':encoder_etiquetas,
    'scaler':scaler,
    'parametros': best_params_nn         
}



joblib.dump(paquete_NN, f'{FOLDER_SALIDA}/models/{model_name}.joblib')

#### <span style="color:#2ca02c">4.7.2. XGBoost usando penultima capa de la red neuronal</span>

In [ ]:
# En Keras 3, cogemos la salida de la penúltima capa y la conectamos directamente a la entrada de la primera capa
capa_latente_output = best_model.layers[-2].output

# Creamos el extractor usando la capa de entrada explícita del modelo
extractor_latente = Model(inputs=best_model.layers[0].input, outputs=capa_latente_output)

# Extraer el espacio latente para tus conjuntos de datos
X_train_latent = extractor_latente.predict(X_train_real)
X_val_latent = extractor_latente.predict(X_val_real)
#X_test_latent = extractor_latente.predict(X_test_real)  # O X_anomalia_test según el nombre que uses

print("¡Espacio latente extraído con éxito! Dimensiones:", X_train_latent.shape)

In [ ]:
print("¡Espacio latente extraído con éxito! Dimensiones:", X_train_latent.shape)

In [ ]:
encoder_etiquetas.inverse_transform(Y_val_real).shape

In [ ]:
# 1. Inicializar el clasificador XGBoost multiclase
xgb_clf = XGBClassifier(
    n_estimators=100,
    learning_rate=0.1,
    max_depth=6,
    random_state=SEED,
    n_jobs=-1
)

# 2. Entrenar XGBoost usando exclusivamente el espacio latente de la red neuronal
print("=== ENTRENANDO XGBOOST CON EL ESPACIO LATENTE ===")
xgb_clf.fit(X_train_latent, Y_train_real)

# 3. Predecir sobre el conjunto de validación o test latente
Y_pred_xgb_encoded = xgb_clf.predict(X_val_latent)

# 4. Revertir a texto original usando tu encoder de etiquetas
Y_pred_xgb_tipos = encoder_etiquetas.inverse_transform(Y_pred_xgb_encoded)
Y_val_real_etiquetas= encoder_etiquetas.inverse_transform(Y_val_real)


# 5. Evaluar resultados
print("\n=== REPORTE DE CLASIFICACIÓN (XGBoost + Espacio Latente) ===")
print(classification_report(Y_val_real_etiquetas, Y_pred_xgb_tipos))

#### <span style="color:#2ca02c">4.7.3. XGBoost usando metricas originales  + penultima capa de la red neuronal</span>

In [ ]:
# 1. Crear el dataset híbrido (Concatenando características crudas + espacio latente de 256 neuronas)
# Asegúrate de usar los mismos conjuntos (aquí usamos train y test para la evaluación final)
X_train_hybrid = np.hstack([X_train_real, X_train_latent])
X_test_hybrid  = np.hstack([X_val_real, X_val_latent]) # O X_val_hybrid si estabas usando validación

# 2. Inicializar el clasificador XGBoost multiclase
xgb_clf = XGBClassifier(
    n_estimators=100,
    learning_rate=0.1,
    max_depth=6,
    random_state=SEED,
    n_jobs=-1
)

# 3. Entrenar XGBoost con el modelo híbrido
print("=== ENTRENANDO XGBOOST HÍBRIDO (Originales + Espacio Latente) ===")
xgb_clf.fit(X_train_hybrid, Y_train_real)

# 4. Predecir sobre el conjunto de test híbrido
Y_pred_xgb_encoded = xgb_clf.predict(X_test_hybrid)

# 5. Revertir a texto original usando el encoder de etiquetas
Y_pred_xgb_tipos = encoder_etiquetas.inverse_transform(Y_pred_xgb_encoded)

# 6. Evaluar resultados frente a las etiquetas reales de test
print("\n=== REPORTE DE CLASIFICACIÓN (XGBoost Híbrido) ===")
print(classification_report(Y_val_real_etiquetas, Y_pred_xgb_tipos))


## <span style="color:#2ca02c">4.8. Clustering no supervisado</span>

### <span style="color:#2ca02c">4.8.1. K-MEANS</span>

In [ ]:
from sklearn.decomposition import PCA
from sklearn.cluster import KMeans

# 1. Reducir la dimensionalidad de tus datos a 2D para poder graficarlos
pca = PCA(n_components=2, random_state=SEED)
X_2d = pca.fit_transform(X_anomalia_test) # O puedes usar X_test_enriquecido

# 2. Aplicar un clustering (por ejemplo, K-Means con un número razonable de grupos visuales, ej. 5 u 8)
n_clusters_visuales = 50
kmeans = KMeans(n_clusters=n_clusters_visuales, random_state=SEED, n_init=10)
clusters = kmeans.fit_predict(X_2d)

# 3. Preparar el DataFrame para la visualización con Seaborn
import pandas as pd
df_plot = pd.DataFrame({
    'PCA1': X_2d[:, 0],
    'PCA2': X_2d[:, 1],
    'Cluster': clusters.astype(str)
})

# 4. Graficar
plt.figure(figsize=(10, 8))
sns.scatterplot(
    data=df_plot, 
    x='PCA1', 
    y='PCA2', 
    hue='Cluster', 
    palette='tab10', 
    alpha=0.6, 
    s=30
)
plt.title(f"Visualización de Clusters en Espacio Reducido (PCA) - {n_clusters_visuales} Grupos")
plt.xlabel("Componente Principal 1")
plt.ylabel("Componente Principal 2")
plt.legend(title="Cluster")
plt.show()

In [ ]:
import matplotlib.pyplot as plt
from sklearn.cluster import KMeans

# 1. Definir un rango razonable de número de clusters a probar (por ejemplo, de 1 a 20)
# Como tienes 53 clases pero quizás se agrupan en menos familias, 20 es un buen tope para buscar el codo.
inercia = []
rango_k = range(1, 21)

for k in rango_k:
    kmeans = KMeans(n_clusters=k, random_state=SEED, n_init=10)
    kmeans.fit(X_anomalia_test) # O puedes usar X_anomalia_train
    inercia.append(kmeans.inertia_)

# 2. Graficar la curva del codo
plt.figure(figsize=(10, 6))
plt.plot(rango_k, inercia, marker='o', linestyle='-', color='b', linewidth=2, markersize=6)
plt.title("Método del Codo para Determinar el Número Óptimo de Clusters", fontsize=14)
plt.xlabel("Número de Clusters (k)", fontsize=12)
plt.ylabel("Inercia (Suma de distancias cuadráticas)", fontsize=12)
plt.xticks(rango_k)
plt.grid(True, linestyle='--', alpha=0.6)
plt.show()

In [ ]:
from sklearn.cluster import KMeans
from sklearn.metrics import silhouette_score
import matplotlib.pyplot as plt

# Rango de clusters a evaluar (por ejemplo, de 2 a 20)
rango_k = range(2, 21)
valores_silueta = []

for k in rango_k:
    kmeans = KMeans(n_clusters=k, random_state=SEED, n_init=10)
    clusters = kmeans.fit_predict(X_anomalia_test)
    
    # Calcular el coeficiente de silueta medio para este k
    score = silhouette_score(X_anomalia_test, clusters)
    valores_silueta.append(score)

# Encontrar el k que maximiza la silueta
k_optimo_silueta = rango_k[valores_silueta.index(max(valores_silueta))]
print(f"Número óptimo de clusters según la Silueta: {k_optimo_silueta}")

# Gráfica opcional para la memoria del TFM
plt.figure(figsize=(9, 5))
plt.plot(rango_k, valores_silueta, marker='s', color='purple', linestyle='-', linewidth=2)
plt.title("Optimización de Clusters por Índice de Silueta")
plt.xlabel("Número de Clusters (k)")
plt.ylabel("Puntuación de Silueta Media")
plt.grid(True, linestyle='--', alpha=0.6)
plt.show()

In [ ]:
import matplotlib.pyplot as plt
import numpy as np
from sklearn.cluster import DBSCAN
from sklearn.metrics import silhouette_score
from sklearn.neighbors import NearestNeighbors

# 1. Encontrar automáticamente un valor óptimo de 'eps' usando la gráfica de k-distancias
# k se suele fijar en min_samples - 1 (por ejemplo, si min_samples = 5, k = 4)
min_samples_val = 5
neighbors = NearestNeighbors(n_neighbors=min_samples_val)
neighbors.fit(X_anomalia_test)
distances, _ = neighbors.kneighbors(X_anomalia_test)

# Ordenar las distancias del k-ésimo vecino más cercano de forma ascendente
k_distances = np.sort(distances[:, min_samples_val - 1])

# Visualizar la curva de k-distancias para buscar el "codo" visualmente si se desea
plt.figure(figsize=(9, 5))
plt.plot(k_distances, color='b', linewidth=2)
plt.title("Gráfica de K-Distancias para encontrar el 'eps' óptimo de DBSCAN")
plt.xlabel("Puntos ordenados por distancia")
plt.ylabel(f"Distancia al {min_samples_val}-ésimo vecino más cercano")
plt.grid(True, linestyle='--', alpha=0.6)
plt.show()

# 2. Automatizar una búsqueda rápida de un 'eps' que devuelva un número razonable de clusters válidos
# Probamos un rango de valores para eps basados en los percentiles de las distancias
mejores_clusters = 0
mejor_eps = 0
mejor_silueta = -1
mejor_etiquetas = None

# Probamos con varios umbrales de distancia
rango_eps = np.percentile(k_distances, np.linspace(50, 95, 10))

for eps_val in rango_eps:
    dbscan = DBSCAN(eps=eps_val, min_samples=min_samples_val)
    etiquetas = dbscan.fit_predict(X_anomalia_test)
    
    # Contar cuántos clusters encontró (excluyendo el ruido, que se marca como -1)
    n_clusters_encontrados = len(set(etiquetas)) - (1 if -1 in etiquetas else 0)
    
    # Para calcular la silueta, DBSCAN requiere que haya al menos 2 clusters y que no todo sea ruido (-1)
    if n_clusters_encontrados > 1:
        # Filtramos temporalmente los puntos de ruido (-1) para calcular la silueta de los clusters reales
        mask_no_ruido = etiquetas != -1
        if sum(mask_no_ruido) > n_clusters_encontrados:
            score = silhouette_score(X_anomalia_test[mask_no_ruido], etiquetas[mask_no_ruido])
            
            # Guardamos la mejor combinación según la silueta
            if score > mejor_silueta:
                mejor_silueta = score
                mejor_eps = eps_val
                mejores_clusters = n_clusters_encontrados
                mejor_etiquetas = etiquetas

print(f"\n--- RESULTADOS ÓPTIMOS DE DBSCAN ---")
print(f"Mejor eps encontrado: {mejor_eps:.4f}")
print(f"Número óptimo de clusters descubiertos: {mejores_clusters}")
print(f"Puntuación de Silueta (sin ruido): {mejor_silueta:.4f}")

# Porcentaje de datos clasificados como ruido (etiqueta -1)
if mejor_etiquetas is not None:
    porcentaje_ruido = (sum(mejor_etiquetas == -1) / len(mejor_etiquetas)) * 100
    print(f"Porcentaje de muestras consideradas ruido (-1): {porcentaje_ruido:.2f}%")

### <span style="color:#2ca02c">4.8.2. DBSCAN</span>

In [ ]:
import matplotlib.pyplot as plt
import numpy as np
from sklearn.cluster import DBSCAN
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import silhouette_score
from sklearn.neighbors import NearestNeighbors

# 1. Estandarizar los datos correctamente
# (Asegúrate de usar el scaler ajustado previamente con tus datos de entrenamiento, 
# o instanciar uno nuevo si es un bloque de análisis exploratorio puro)
scaler = StandardScaler()
X_scaled = scaler.fit_transform(X_anomalia_test)

# 2. Encontrar automáticamente un valor óptimo de 'eps' usando la gráfica de k-distancias
min_samples_val = 5
neighbors = NearestNeighbors(n_neighbors=min_samples_val)
neighbors.fit(X_scaled)
distances, _ = neighbors.kneighbors(X_scaled)

# Ordenar las distancias del k-ésimo vecino más cercano
k_distances = np.sort(distances[:, min_samples_val - 1])

# Visualizar la curva de k-distancias para verificar el "codo" con los datos escalados
plt.figure(figsize=(9, 5))
plt.plot(k_distances, color='b', linewidth=2)
plt.title("Gráfica de K-Distancias (Datos Estandarizados)")
plt.xlabel("Puntos ordenados por distancia")
plt.ylabel(f"Distancia al {min_samples_val}-ésimo vecino más cercano")
plt.grid(True, linestyle='--', alpha=0.6)
plt.show()

# 3. Búsqueda automatizada del mejor 'eps' basada en la silueta y un nivel de ruido razonable
mejores_clusters = 0
mejor_eps = 0
mejor_silueta = -1
mejor_etiquetas = None

# Probamos con varios percentiles del rango de distancias
rango_eps = np.percentile(k_distances, np.linspace(50, 95, 15))

for eps_val in rango_eps:
    dbscan = DBSCAN(eps=eps_val, min_samples=min_samples_val)
    etiquetas = dbscan.fit_predict(X_scaled)
    
    # Contar clusters encontrados (excluyendo el ruido -1)
    n_clusters_encontrados = len(set(etiquetas)) - (1 if -1 in etiquetas else 0)
    
    if n_clusters_encontrados > 1:
        mask_no_ruido = etiquetas != -1
        if sum(mask_no_ruido) > n_clusters_encontrados:
            score = silhouette_score(X_scaled[mask_no_ruido], etiquetas[mask_no_ruido])
            
            if score > mejor_silueta:
                mejor_silueta = score
                mejor_eps = eps_val
                mejores_clusters = n_clusters_encontrados
                mejor_etiquetas = etiquetas

print(f"\n--- RESULTADOS DBSCAN (CON DATOS ESTANDARIZADOS) ---")
print(f"Mejor eps encontrado: {mejor_eps:.4f}")
print(f"Número óptimo de clusters descubiertos: {mejores_clusters}")
print(f"Puntuación de Silueta (sin ruido): {mejor_silueta:.4f}")

if mejor_etiquetas is not None:
    porcentaje_ruido = (sum(mejor_etiquetas == -1) / len(mejor_etiquetas)) * 100
    print(f"Porcentaje de muestras consideradas ruido (-1): {porcentaje_ruido:.2f}%")

In [ ]:
import matplotlib.pyplot as plt
import numpy as np
from sklearn.cluster import DBSCAN
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import silhouette_score
from sklearn.neighbors import NearestNeighbors
from kneed import KneeLocator

# 1. Estandarizar los datos correctamente
scaler = StandardScaler()
X_scaled = scaler.fit_transform(X_anomalia_test)

# 2. Encontrar las k-distancias para el cálculo automático
min_samples_val = 5
neighbors = NearestNeighbors(n_neighbors=min_samples_val)
neighbors.fit(X_scaled)
distances, _ = neighbors.kneighbors(X_scaled)

# Ordenar las distancias del k-ésimo vecino más cercano
k_distances = np.sort(distances[:, min_samples_val - 1])
puntos_x = np.arange(len(k_distances))

# 3. Detectar automáticamente el "codo" en la curva de K-distancias usando KneeLocator
kneedle = KneeLocator(
    puntos_x, 
    k_distances, 
    curve='convex', 
    direction='increasing'
)

eps_optimo = kneedle.elbow_y

# Si por algún motivo el localizador automático no devolviera un valor, usamos un percentil de seguridad
if eps_optimo is None:
    eps_optimo = np.percentile(k_distances, 95)

print(f"--- CÁLCULO AUTOMÁTICO ---")
print(f"Valor de 'eps' óptimo detectado en el codo: {eps_optimo:.4f}")

# 4. Ejecutar DBSCAN con el 'eps' calculado automáticamente
dbscan = DBSCAN(eps=eps_optimo, min_samples=min_samples_val)
etiquetas = dbscan.fit_predict(X_scaled)

# 5. Analizar los resultados obtenidos
n_clusters_encontrados = len(set(etiquetas)) - (1 if -1 in etiquetas else 0)
porcentaje_ruido = (sum(etiquetas == -1) / len(etiquetas)) * 100

print(f"\n--- RESULTADOS FINALES DE DBSCAN ---")
print(f"Número de clusters identificados: {n_clusters_encontrados}")
print(f"Porcentaje de muestras consideradas ruido (-1): {porcentaje_ruido:.2f}%")

if n_clusters_encontrados > 1:
    mask_no_ruido = etiquetas != -1
    if sum(mask_no_ruido) > n_clusters_encontrados:
        score_silueta = silhouette_score(X_scaled[mask_no_ruido], etiquetas[mask_no_ruido])
        print(f"Puntuación de Silueta (sin ruido): {score_silueta:.4f}")

# 6. Mostrar la gráfica señalando el codo detectado
plt.figure(figsize=(9, 5))
plt.plot(puntos_x, k_distances, color='b', linewidth=2, label='K-distancias')
if kneedle.elbow is not None:
    plt.axvline(x=kneedle.elbow, color='r', linestyle='--', label=f'Codo automático (eps={eps_optimo:.2f})')
plt.title("Gráfica de K-Distancias con Codo Automático (DBSCAN)")
plt.xlabel("Puntos ordenados por distancia")
plt.ylabel(f"Distancia al {min_samples_val}-ésimo vecino más cercano")
plt.legend()
plt.grid(True, linestyle='--', alpha=0.6)
plt.show()

In [ ]:
import pandas as pd
from sklearn.metrics import adjusted_rand_score, normalized_mutual_info_score

# 1. Obtener el vector de clasificación (etiquetas predichas por DBSCAN)
# 'etiquetas' es el array que devolvió el modelo DBSCAN (valores de -1 a 7)
vector_clusters = etiquetas

# 2. Crear un DataFrame analítico cruzando los clusters con tus etiquetas reales
df_cruce = pd.DataFrame({
    'Cluster_DBSCAN': vector_clusters,
    'Etiqueta_Real': Y_test_etiquetas  # Asegúrate de que esta variable contenga tus 53 clases reales
})

# 3. Generar la Matriz de Contingencia (Tabla cruzada)
# Filas = Clusters de DBSCAN (incluyendo el -1 como ruido), Columnas = Tus 53 clases reales (o puedes ver una muestra)
matriz_contingencia = pd.crosstab(df_cruce['Cluster_DBSCAN'], df_cruce['Etiqueta_Real'])

print("--- MATRIZ DE CONTINGENCIA (Clusters vs Clases Reales) ---")
print("Dimensiones de la matriz:", matriz_contingencia.shape)
# Mostramos una pequeña porción para visualizarlo cómodamente en consola
print(matriz_contingencia.iloc[:, :]) 

# 4. Calcular métricas de relación / concordancia externa
# (Nota: Omitimos temporalmente el ruido -1 para evaluar qué tan bien agrupados están los puntos válidos)
mask_evaluacion = vector_clusters != -1
if sum(mask_evaluacion) > 0:
    y_real_filtrado = Y_test_etiquetas[mask_evaluacion]
    y_pred_filtrado = vector_clusters[mask_evaluacion]

    ari = adjusted_rand_score(y_real_filtrado, y_pred_filtrado)
    nmi = normalized_mutual_info_score(y_real_filtrado, y_pred_filtrado)

    print(f"\n--- MÉTRICAS DE CONCORDANCIA EXTERNA ---")
    print(f"Adjusted Rand Index (ARI): {ari:.4f}  (0 = azar, 1 = coincidencia perfecta)")
    print(f"Normalized Mutual Information (NMI): {nmi:.4f}  (Mide cuánta información comparten)")

# <span style="color:#2ca02c">5. Revision complementariedad de modelos</span>

In [ ]:
def calcular_potencial_mejora(y_true, y_pred_1, y_pred_2, average='macro'):
    """
    Calcula un valor numérico que indica si un modelo híbrido mejorará 
    el rendimiento de los modelos individuales.
    """
    y_true = np.array(y_true)
    y_pred_1 = np.array(y_pred_1)
    y_pred_2 = np.array(y_pred_2)
    
    # 1. Métricas base de los modelos individuales (ej. Macro F1)
    score_1 = f1_score(y_true, y_pred_1, average=average, zero_division=0)
    score_2 = f1_score(y_true, y_pred_2, average=average, zero_division=0)
    mejor_base = max(score_1, score_2)
    
    # 2. Techo Teórico (Oracle): Simulación perfecta donde elegimos el acierto 
    # de cualquiera de los dos modelos si al menos uno acierta la muestra.
    aciertos_1 = (y_pred_1 == y_true)
    aciertos_2 = (y_pred_2 == y_true)
    acierto_or = aciertos_1 | aciertos_2
    
    # Si simulamos que un hipotético híbrido perfecto rescata todos los aciertos posibles:
    # (Para muestras donde ambos fallan, mantenemos una predicción al azar o del mejor)
    # Una aproximación numérica directa del límite superior (Techo Oracle):
    techo_teorico = np.mean(acierto_or) # (Si usamos accuracy como base de techo)
    
    # 3. CÁLCULO DEL NÚMERO CLAVE: "Margen de Mejora Potencial" (Delta)
    # Mide cuántos puntos porcentuales hay de margen entre el mejor modelo y el límite teórico.
    # (Lo calculamos con el acierto binario general o adaptado al F1)
    acc_1 = accuracy_score(y_true, y_pred_1)
    acc_2 = accuracy_score(y_true, y_pred_2)
    mejor_acc_base = max(acc_1, acc_2)
    
    delta_mejora = techo_teorico - mejor_acc_base
    
    # 4. Tasa de Desacuerdo (Diversidad)
    tasa_desacuerdo = np.mean(aciertos_1 != aciertos_2)
    
    print("=" * 60)
    print("      INDICADOR NUMÉRICO DE POTENCIAL DE MEJORA")
    print("=" * 60)
    print(f"Score (Macro F1) Modelo 1:    {score_1:.4f}")
    print(f"Score (Macro F1) Modelo 2:    {score_2:.4f}")
    print(f"Mejor Modelo Individual:      {mejor_base:.4f}")
    print("-" * 60)
    print(f"Tasa de Desacuerdo (Diversidad): {tasa_desacuerdo * 100:.2f}%")
    print(f"Margen de Mejora Teórico (Delta): +{delta_mejora * 100:.2f}%  <-- NÚMERO CLAVE")
    print("=" * 60)
    
    # Interpretación numérica estricta:
    if delta_mejora > 0.03:  # Más de un 3% de margen teórico
        print("🟢 Veredicto: ALTO POTENCIAL DE MEJORA (> 3% de margen).")
    elif delta_mejora > 0.01: # Entre 1% y 3%
        print("🟡 Veredicto: MEJORA MODERADA O MARGINAL (1% - 3%).")
    else:
        print("🔴 Veredicto: SIN POTENCIAL DE MEJORA (< 1%). Los modelos son redundantes.")
        
    return {
        'mejor_base': mejor_base,
        'techo_teorico': techo_teorico,
        'delta_mejora': delta_mejora,
        'tasa_desacuerdo': tasa_desacuerdo
    }

In [95]:
lista_filenames=[
    'result/models/RandomForest (metricas_logs 3 puntos)(todas las features).joblib',
    'result/models/RandomForest (metricas_logs 3 puntos)   N_feat=63.joblib',
    
    'result/models/RandomForest (metricas_logs 1 punto suma)(todas las features).joblib',
    'result/models/RandomForest (metricas_logs 1 punto suma)   N_feat=78.joblib',
    
    'result/models/XGBoost (metricas_logs 3 puntos)(todas las features).joblib',
    'result/models/XGBoost (metricas_logs 3 puntos)   N_feat=102.joblib',
    
    'result/models/XGBoost (metricas_logs 1 punto suma)(todas las features).joblib',
    'result/models/XGBoost (metricas_logs 1 punto suma)   N_feat=160.joblib',
    
    'result/models/LightGBM (metricas_logs 3 puntos)(todas las features).joblib',
    'result/models/LightGBM (metricas_logs 3 puntos)   N_feat=147.joblib',
    
    'result/models/LightGBM (metricas_logs 1 punto suma)(todas las features).joblib',
    'result/models/LightGBM (metricas_logs 1 punto suma)   N_feat=202.joblib',
    
#     'result/models/RandomForest (metricas_log) N_feat=44.joblib',
#     'result/models/XGBoost con metricas_logs (3 puntos y todas las metricas).joblib',
#     'result/models/XGBoost (metricas_log) N_feat=93.joblib',
#    'result/models/XGBoost con metricas_logs (1 punto 3 intervalos todas las metricas).joblib'
#     'result/models/XGBoost (metricas_log) N_feat=110.joblib'
]

In [96]:
paquete_modelo['resultado']

{'accuracy': 0.6882438316400581,
 'precision': 0.6910796659371462,
 'recall': 0.6882438316400581,
 'f1': 0.6878356873422209}

In [97]:
lista_modelos=[]
for filename in lista_filenames:
    paquete_modelo = joblib.load(filename)
    lista_modelos.append(paquete_modelo)
    


In [98]:
for paquete_modelo in lista_modelos:
    model_name      = paquete_modelo['model_name']
    accuracy        = paquete_modelo['resultado']['accuracy']
    macro_f1        = paquete_modelo['resultado']['precision']
    macro_precision = paquete_modelo['resultado']['recall']
    macro_recall    = paquete_modelo['resultado']['f1']
    print(f"Model = {model_name:70s} Accuracy = {accuracy:.4f} | Macro F1 = {macro_f1:.4f} | Macro Precision = {macro_precision:.4f} | Macro Recall = {macro_recall:.4f}")


Model = RandomForest (metricas_logs 3 puntos)(todas las features)              Accuracy = 0.6232 | Macro F1 = 0.6255 | Macro Precision = 0.6232 | Macro Recall = 0.6192
Model = RandomForest (metricas_logs 3 puntos)   N_feat=63                      Accuracy = 0.6374 | Macro F1 = 0.6414 | Macro Precision = 0.6374 | Macro Recall = 0.6352
Model = RandomForest (metricas_logs 1 punto suma)(todas las features)          Accuracy = 0.6136 | Macro F1 = 0.6076 | Macro Precision = 0.6136 | Macro Recall = 0.6008


KeyError: 'precision'

In [115]:
paquete_modelo['resultado']

{'accuracy': 0.6275761973875181,
 'macro_f1': 0.6197418658287879,
 'macro_precision': 0.6283339292003264,
 'macro_recall': 0.6275761973875182}

In [ ]:
lista_Y_pred=[]
Y_pred =  predict_modelo(lista_modelos[0],lista_feature_names1_sum,X_test_sum)
lista_Y_pred.append(Y_pred)
Y_pred =  predict_modelo(lista_modelos[1],lista_feature_names1,X_test)
lista_Y_pred.append(Y_pred)
# Y_pred_tipos                                = encoder_etiquetas.inverse_transform(Y_pred)

In [ ]:
calcular_potencial_mejora(Y_test_encoded,lista_Y_pred[0],lista_Y_pred[1])

In [ ]:

from sklearn.linear_model import LogisticRegression

# 1. Obtener las matrices de validación con las features específicas de cada modelo
X_val_sub_1 = X_matrix_sel_data(X_matrix, lista_feature_names, lista_idx_filas=idx_val, lista_feature_sel=lista_modelos[0]['lista_features'])
X_val_sub_2 = X_matrix_sel_data(X_matrix, lista_feature_names, lista_idx_filas=idx_val, lista_feature_sel=lista_modelos[1]['lista_features'])

# 2. Generar las probabilidades sobre ESE conjunto de validación (datos no vistos)
probs_val_1 = lista_modelos[0]['model'].predict_proba(X_val_sub_1)
probs_val_2 = lista_modelos[1]['model'].predict_proba(X_val_sub_2)

# 3. Construir las meta-features de entrenamiento para el metamodelo
X_meta_train = np.hstack((probs_val_1, probs_val_2))

# 4. Entrenar el metamodelo
meta_model = LogisticRegression(max_iter=1000,class_weight='balanced')
meta_model.fit(X_meta_train, Y_val_encoded)


X_test_sub_1 = X_matrix_sel_data(X_matrix, lista_feature_names, lista_idx_filas=idx_test, lista_feature_sel=lista_modelos[0]['lista_features'])
X_test_sub_2 = X_matrix_sel_data(X_matrix, lista_feature_names, lista_idx_filas=idx_test, lista_feature_sel=lista_modelos[1]['lista_features'])

probs_test_1 = lista_modelos[0]['model'].predict_proba(X_test_sub_1)
probs_test_2 = lista_modelos[1]['model'].predict_proba(X_test_sub_2)

# 5. Para test (haces lo mismo pero con X_test)
X_meta_test = np.hstack((probs_test_1, probs_test_2))
y_pred_meta = meta_model.predict(X_meta_test)

In [ ]:
Y_pred_tipos                                = encoder_etiquetas.inverse_transform(y_pred_meta)

In [ ]:
print(classification_report(Y_test_etiquetas, Y_pred_tipos))